# 🚀 Supervisely ML Training Studio

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LorenzFon/supervisely-colab-trainer/blob/main/supervisely_training.ipynb)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)
[![Python 3.10+](https://img.shields.io/badge/python-3.10+-blue.svg)](https://www.python.org/downloads/)

An end-to-end, zero-restart interactive GUI studio for Google Colab that connects directly to your [Supervisely](https://supervisely.com) workspace. Select projects from an interactive dropdown, automatically convert annotations, and train state-of-the-art object detection models across **10 model families and 54 checkpoints** (**RT-DETRv2**, **DEIMv2**, **MMDetection**, **YOLOX**, **YOLO11**, **YOLOv10**, **YOLOv9**, **YOLOv8**, **YOLOv5u**, and **YOLO-World v2**).

### ✨ Key Features
- **⚡ Zero-Restart Setup:** Memory-mapped package pinning prevents Colab runtime restarts.
- **🔄 Auto Project & Validation Discovery:** Browse Supervisely projects via dropdown; automatically detects dedicated `validation` / `val` datasets and locks the split slider.
- **📜 License Transparency:** Displays open-source licensing (`Apache 2.0` permissive vs. `AGPL-3.0 / GPL-3.0`) directly on every model family card.
- **📊 Live Visual Dashboard & Accurate ETA:** Real-time KPI tiles for Current Epoch, Train Loss, Val Loss / mAP50, and **Remaining Time (ETA)**.
- **📈 Collapsible Loss Convergence Drill:** Interactive vector chart plotting live Training & Validation curves inside a compact, collapsible drill-down drawer.
- **⏹️ Graceful Stop & Non-Blocking UI:** Background daemon thread keeps the UI responsive so you can switch between Visual Dashboard and Raw Console or stop training cleanly at any moment.
- **📱 Automated Edge AI & On-Device Exports:**
  - **Apple CoreML** (`.mlpackage`) with NMS for iOS, iPadOS, macOS
  - **Google LiteRT** (`.tflite`) for Android, Raspberry Pi, Coral, and on-device AI
  - **ONNX Runtime** (`.onnx`) for cross-platform and web deployment
- **💾 One-Click Bundle Export:** Download trained weights and all converted edge models packaged in a single `.zip` file with duplicate-download protection.

---
### 🏁 Quick Start
1. Ensure your Colab runtime is set to **GPU** (`Runtime` &rarr; `Change runtime type` &rarr; `T4 GPU`).
2. Run the cell below by clicking the **Play button (▶️)**.
3. Enter your Supervisely API Token (or configure `SUPERVISELY_TOKEN` under Colab Secrets 🔑) and click **🔄 Fetch My Projects**.
4. Select your project, choose your model architecture & export formats, and click **🚀 Start Training**.


In [ ]:
# @title ▶️ Launch Supervisely ML Training Studio { display-mode: "form" }
import os
import sys
import math
import subprocess
import random
import json
import re
import shutil
import time
import threading
from pathlib import Path
from IPython.display import display, HTML, clear_output

# --- 1. FULLY AUTOMATED ZERO-RESTART DEPENDENCY LOADER ---
try:
    import supervisely as sly
except (ImportError, ModuleNotFoundError):
    print("⏳ Setting up Supervisely automatically (Zero-Restart Mode)... Please wait ~60 seconds.")
    try:
        import PIL
        pillow_constraint = f"pillow=={PIL.__version__}"
    except Exception:
        pillow_constraint = "pillow>=10.4.0,<11.2.0"

    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", pillow_constraint, "supervisely"
    ])
    for mod in list(sys.modules.keys()):
        if mod == "PIL" or mod.startswith("PIL."):
            del sys.modules[mod]
    import supervisely as sly
    clear_output(wait=True)

import torch
from PIL import Image
import ipywidgets as widgets

# Fetch token from Colab Secrets if configured
default_token = ""
try:
    from google.colab import userdata
    default_token = userdata.get("SUPERVISELY_TOKEN") or ""
except Exception:
    default_token = ""

# --- 2. MODEL CATALOG & PER-MODEL DEPENDENCIES ---
MODEL_CATALOG = {
    "RT-DETRv2 (Hugging Face Transformers)": {
        "license": "Apache 2.0 (Permissive / Commercial Friendly)",
        "license_pill": "<span style='background:#fef9c3; color:#854d0e; padding:3px 8px; border-radius:10px; font-weight:700; font-size:11px;'>📜 License: Apache 2.0 (Commercial Friendly)</span>",
        "packages": ["transformers>=4.43.0", "accelerate", "pycocotools", "torchmetrics"],
        "badge": "⚡ <b>RT-DETRv2 (Vision Transformer):</b> End-to-end real-time detector without NMS. Best for PyTorch, ONNX, and CoreML (Note: LiteRT/TFLite lacks transformer deformable attention support; use YOLO11 for LiteRT).",
        "badge_style": "background:#eff6ff; border:1px solid #bfdbfe; color:#1d4ed8;",
        "variants": [
            ("RT-DETRv2-S (ResNet-18 | Fast & Lightweight)", "PekingU/rtdetr_v2_r18vd"),
            ("RT-DETRv2-M (ResNet-34 | Compact)", "PekingU/rtdetr_v2_r34vd"),
            ("RT-DETRv2-L (ResNet-50 | Recommended)", "PekingU/rtdetr_v2_r50vd"),
            ("RT-DETRv2-X (ResNet-101 | High Accuracy)", "PekingU/rtdetr_v2_r101vd"),
        ]
    },
    "DEIMv2 (DINOv3 / Real-Time DETR)": {
        "license": "Apache 2.0 (Permissive / Commercial Friendly)",
        "license_pill": "<span style='background:#fef9c3; color:#854d0e; padding:3px 8px; border-radius:10px; font-weight:700; font-size:11px;'>📜 License: Apache 2.0 (Commercial Friendly)</span>",
        "packages": ["torch", "torchvision", "pycocotools", "huggingface_hub", "safetensors", "pyyaml", "scipy", "faster-coco-eval"],
        "badge": "⚡ <b>DEIMv2:</b> Real-Time Object Detection Meets DINOv3. Combines DINOv3 representation with Dense O2O matching and pruned HGNetv2 edge backbones. Permissive Apache 2.0 license.",
        "badge_style": "background:#f5f3ff; border:1px solid #ddd6fe; color:#6d28d9;",
        "variants": [
            ("DEIMv2 Atto (HGNetv2 | 0.49M Params / Micro-Edge)", "Intellindust/DEIMv2_HGNetv2_ATTO_COCO"),
            ("DEIMv2 Femto (HGNetv2 | 0.96M Params / Ultra-Light)", "Intellindust/DEIMv2_HGNetv2_FEMTO_COCO"),
            ("DEIMv2 Pico (HGNetv2 | 1.5M Params / Mobile & IoT)", "Intellindust/DEIMv2_HGNetv2_PICO_COCO"),
            ("DEIMv2 Nano (HGNetv2 | 3.2M Params / High Speed)", "Intellindust/DEIMv2_HGNetv2_N_COCO"),
            ("DEIMv2 Small (DINOv3-Tiny | 9.7M Params / SOTA Speed-Accuracy)", "Intellindust/DEIMv2_DINOv3_S_COCO"),
            ("DEIMv2 Medium (DINOv3-Tiny+ | 18.5M Params / High Accuracy)", "Intellindust/DEIMv2_DINOv3_M_COCO"),
            ("DEIMv2 Large (DINOv3-ViT | 33.2M Params / Heavy Duty)", "Intellindust/DEIMv2_DINOv3_L_COCO"),
            ("DEIMv2 Extra-Large (DINOv3-ViT+ | 50.3M Params / Top Precision)", "Intellindust/DEIMv2_DINOv3_X_COCO"),
        ]
    },
    "MMDetection (OpenMMLab)": {
        "license": "Apache 2.0 (Permissive / Commercial Friendly)",
        "license_pill": "<span style='background:#fef9c3; color:#854d0e; padding:3px 8px; border-radius:10px; font-weight:700; font-size:11px;'>📜 License: Apache 2.0 (Commercial Friendly)</span>",
        "packages": ["openmim", "mmengine", "mmcv>=2.0.0", "mmdet>=3.0.0"],
        "badge": "🔬 <b>MMDetection:</b> OpenMMLab's modular detection benchmark featuring RTMDet and Faster R-CNN. Fully permissive Apache 2.0 license for commercial deployment.",
        "badge_style": "background:#f0f9ff; border:1px solid #bae6fd; color:#0369a1;",
        "variants": [
            ("RTMDet-Tiny (4.8M Params | Ultra-Fast Edge)", "rtmdet_tiny_8xb32-300e_coco"),
            ("RTMDet-Small (8.9M Params | Balanced)", "rtmdet_s_8xb32-300e_coco"),
            ("RTMDet-Medium (24.7M Params | High Accuracy)", "rtmdet_m_8xb32-300e_coco"),
            ("RTMDet-Large (52.3M Params | Heavy Duty)", "rtmdet_l_8xb32-300e_coco"),
            ("RTMDet-ExtraLarge (94.9M Params | Top Precision)", "rtmdet_x_8xb32-300e_coco"),
            ("Faster R-CNN (ResNet-50 | Two-Stage Industry Classic)", "faster-rcnn_r50_fpn_1x_coco"),
        ]
    },
    "YOLOX (Megvii)": {
        "license": "Apache 2.0 (Permissive / Commercial Friendly)",
        "license_pill": "<span style='background:#fef9c3; color:#854d0e; padding:3px 8px; border-radius:10px; font-weight:700; font-size:11px;'>📜 License: Apache 2.0 (Commercial Friendly)</span>",
        "packages": ["tabulate", "loguru", "pycocotools", "thop"],
        "badge": "🚀 <b>YOLOX (Megvii):</b> High-performance anchor-free detector with decoupled head and SimOTA label assignment. Fully permissive Apache 2.0 license for commercial deployment without copyleft restrictions.",
        "badge_style": "background:#f0fdf4; border:1px solid #86efac; color:#166534;",
        "variants": [
            ("YOLOX-Nano (yolox_nano.pth | 0.91M Params / Edge)", "yolox_nano.pth"),
            ("YOLOX-Tiny (yolox_tiny.pth | 5.06M Params / Mobile)", "yolox_tiny.pth"),
            ("YOLOX-S (yolox_s.pth | 9.0M Params / Balanced)", "yolox_s.pth"),
            ("YOLOX-M (yolox_m.pth | 25.3M Params / High Accuracy)", "yolox_m.pth"),
            ("YOLOX-L (yolox_l.pth | 54.2M Params / Heavy Duty)", "yolox_l.pth"),
            ("YOLOX-X (yolox_x.pth | 99.1M Params / Top Precision)", "yolox_x.pth"),
        ]
    },
    "YOLO11 (Ultralytics)": {
        "license": "AGPL-3.0 (Copyleft / Commercial License via Ultralytics)",
        "license_pill": "<span style='background:#fee2e2; color:#991b1b; padding:3px 8px; border-radius:10px; font-weight:700; font-size:11px;'>📜 License: AGPL-3.0 (Ultralytics / Commercial)</span>",
        "packages": ["ultralytics", "pyyaml"],
        "badge": "🚀 <b>YOLO11:</b> Latest Ultralytics state-of-the-art detector. 100% native support for Google LiteRT (.tflite), Apple CoreML (.mlpackage), ONNX, and PyTorch.",
        "badge_style": "background:#f0fdf4; border:1px solid #bbf7d0; color:#15803d;",
        "variants": [
            ("YOLO11 Nano (yolo11n.pt | Ultra-Fast Edge)", "yolo11n.pt"),
            ("YOLO11 Small (yolo11s.pt | Balanced)", "yolo11s.pt"),
            ("YOLO11 Medium (yolo11m.pt | High Accuracy)", "yolo11m.pt"),
            ("YOLO11 Large (yolo11l.pt | Heavy Duty)", "yolo11l.pt"),
            ("YOLO11 Extra-Large (yolo11x.pt | Maximum Precision)", "yolo11x.pt"),
        ]
    },
    "YOLOv10 (Ultralytics / Tsinghua)": {
        "license": "AGPL-3.0 (NMS-Free End-to-End Detector)",
        "license_pill": "<span style='background:#fee2e2; color:#991b1b; padding:3px 8px; border-radius:10px; font-weight:700; font-size:11px;'>📜 License: AGPL-3.0 (Ultralytics / Commercial)</span>",
        "packages": ["ultralytics", "pyyaml"],
        "badge": "⚡ <b>YOLOv10 (NMS-Free):</b> Dual-label assignments eliminate post-processing Non-Maximum Suppression (NMS) latency. Ideal for real-time mobile & edge inference.",
        "badge_style": "background:#ecfeff; border:1px solid #a5f3fc; color:#0e7490;",
        "variants": [
            ("YOLOv10 Nano (yolov10n.pt | Sub-3M Params)", "yolov10n.pt"),
            ("YOLOv10 Small (yolov10s.pt | Fast)", "yolov10s.pt"),
            ("YOLOv10 Medium (yolov10m.pt | Compact)", "yolov10m.pt"),
            ("YOLOv10 Balanced (yolov10b.pt | Balanced Speed/mAP)", "yolov10b.pt"),
            ("YOLOv10 Large (yolov10l.pt | High Precision)", "yolov10l.pt"),
            ("YOLOv10 Extra-Large (yolov10x.pt | Top Accuracy)", "yolov10x.pt"),
        ]
    },
    "YOLOv9 (Ultralytics / PGI & GELAN)": {
        "license": "AGPL-3.0 / GPL-3.0 (Programmable Gradient Info)",
        "license_pill": "<span style='background:#fee2e2; color:#991b1b; padding:3px 8px; border-radius:10px; font-weight:700; font-size:11px;'>📜 License: AGPL-3.0 / GPL-3.0</span>",
        "packages": ["ultralytics", "pyyaml"],
        "badge": "🧠 <b>YOLOv9 (PGI & GELAN):</b> Retains rich gradient information across deep layers using Programmable Gradient Information. Exceptional mAP on complex multi-class datasets.",
        "badge_style": "background:#fdf4ff; border:1px solid #f0abfc; color:#a21caf;",
        "variants": [
            ("YOLOv9 Tiny (yolov9t.pt | Lightweight)", "yolov9t.pt"),
            ("YOLOv9 Small (yolov9s.pt | Fast)", "yolov9s.pt"),
            ("YOLOv9 Medium (yolov9m.pt | Accurate)", "yolov9m.pt"),
            ("YOLOv9 Compact (yolov9c.pt | High Capacity)", "yolov9c.pt"),
            ("YOLOv9 Extended (yolov9e.pt | Maximum Power)", "yolov9e.pt"),
        ]
    },
    "YOLOv8 (Ultralytics)": {
        "license": "AGPL-3.0 (Copyleft / Commercial License via Ultralytics)",
        "license_pill": "<span style='background:#fee2e2; color:#991b1b; padding:3px 8px; border-radius:10px; font-weight:700; font-size:11px;'>📜 License: AGPL-3.0 (Ultralytics / Commercial)</span>",
        "packages": ["ultralytics", "pyyaml"],
        "badge": "🎯 <b>YOLOv8:</b> Industry-standard anchor-free detector with mature production ecosystem across TensorRT, OpenVINO, CoreML, and LiteRT.",
        "badge_style": "background:#faf5ff; border:1px solid #e9d5ff; color:#7e22ce;",
        "variants": [
            ("YOLOv8 Nano (yolov8n.pt | Fast Edge)", "yolov8n.pt"),
            ("YOLOv8 Small (yolov8s.pt | Balanced)", "yolov8s.pt"),
            ("YOLOv8 Medium (yolov8m.pt | High Accuracy)", "yolov8m.pt"),
            ("YOLOv8 Large (yolov8l.pt | Heavy Duty)", "yolov8l.pt"),
            ("YOLOv8 Extra-Large (yolov8x.pt | Maximum Precision)", "yolov8x.pt"),
        ]
    },
    "YOLOv5u (Ultralytics)": {
        "license": "AGPL-3.0 (Ultralytics Production Workhorse)",
        "license_pill": "<span style='background:#fee2e2; color:#991b1b; padding:3px 8px; border-radius:10px; font-weight:700; font-size:11px;'>📜 License: AGPL-3.0 (Ultralytics / Commercial)</span>",
        "packages": ["ultralytics", "pyyaml"],
        "badge": "🏭 <b>YOLOv5u:</b> Modernized anchor-free split-head YOLOv5. Unrivaled compatibility across legacy industrial cameras, older embedded devices, and production chips.",
        "badge_style": "background:#fff7ed; border:1px solid #fed7aa; color:#c2410c;",
        "variants": [
            ("YOLOv5u Nano (yolov5nu.pt | Microcontrollers & Edge)", "yolov5nu.pt"),
            ("YOLOv5u Small (yolov5su.pt | Fast)", "yolov5su.pt"),
            ("YOLOv5u Medium (yolov5mu.pt | Production Standard)", "yolov5mu.pt"),
            ("YOLOv5u Large (yolov5lu.pt | High Accuracy)", "yolov5lu.pt"),
            ("YOLOv5u Extra-Large (yolov5xu.pt | Maximum Capacity)", "yolov5xu.pt"),
        ]
    },
    "YOLO-World v2 (Ultralytics / Open-Vocabulary)": {
        "license": "GPL-3.0 / AGPL-3.0 (Vision-Language Zero-Shot & Fine-Tuning)",
        "license_pill": "<span style='background:#fee2e2; color:#991b1b; padding:3px 8px; border-radius:10px; font-weight:700; font-size:11px;'>📜 License: GPL-3.0 / AGPL-3.0</span>",
        "packages": ["ultralytics", "pyyaml"],
        "badge": "🌐 <b>YOLO-World v2:</b> Real-time vision-language open-vocabulary detector. Pre-trained on massive multimodal datasets; highly effective when fine-tuned on custom Supervisely annotations.",
        "badge_style": "background:#f0fdfa; border:1px solid #99f6e4; color:#0f766e;",
        "variants": [
            ("YOLO-Worldv2 Small (yolov8s-worldv2.pt | Fast)", "yolov8s-worldv2.pt"),
            ("YOLO-Worldv2 Medium (yolov8m-worldv2.pt | Balanced)", "yolov8m-worldv2.pt"),
            ("YOLO-Worldv2 Large (yolov8l-worldv2.pt | High Precision)", "yolov8l-worldv2.pt"),
            ("YOLO-Worldv2 Extra-Large (yolov8x-worldv2.pt | Top Accuracy)", "yolov8x-worldv2.pt"),
        ]
    },
}

# --- 3. UI DASHBOARD COMPONENTS ---
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "⚠️ No GPU detected (Runtime -> Change runtime type to GPU)"
gpu_badge_color = "#16a34a" if torch.cuda.is_available() else "#dc2626"

header_html = widgets.HTML(f"""
<div style="background: linear-gradient(135deg, #1e293b, #0f172a); color: white; padding: 18px 22px; border-radius: 12px; margin-bottom: 12px; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1);">
    <div style="display:flex; justify-content:space-between; align-items:center; flex-wrap: wrap; gap: 8px;">
        <div>
            <h2 style="margin: 0 0 4px 0; font-size: 20px; color: #f8fafc; font-weight: 700; letter-spacing: -0.02em;">🚀 Supervisely ML Training Studio</h2>
            <p style="margin: 0; color: #94a3b8; font-size: 13px;">Live Visual Dashboard, real-time loss graphs, ETA calculation, and multi-format export.</p>
        </div>
        <div style="background: {gpu_badge_color}; color: white; padding: 6px 14px; border-radius: 20px; font-size: 12px; font-weight: 600; box-shadow: 0 2px 4px rgba(0,0,0,0.2);">
            🖥️ {gpu_name}
        </div>
    </div>
    <div style="margin-top: 10px; padding-top: 8px; border-top: 1px solid #334155; font-size: 11px; color: #cbd5e1;">
        💡 <b>Tip:</b> Hover your mouse over any input, button, or slider to view helpful tooltips and best-practice guidance.
    </div>
</div>
""")

style_label = {"description_width": "150px"}
full_layout = widgets.Layout(width="98%")

# Section 1: Supervisely Connection & Auto-Discovered Projects
server_input = widgets.Text(
    value="https://app.supervisely.com",
    description="🌐 Server URL:",
    tooltip="Supervisely instance URL (default: https://app.supervisely.com or custom enterprise server address)",
    style=style_label, layout=full_layout
)
token_input = widgets.Password(
    value=default_token,
    placeholder="Enter Supervisely API token (or set via Colab Secret SUPERVISELY_TOKEN)",
    description="🔑 API Token:",
    tooltip="Your personal Supervisely API token (found in Supervisely under Account Settings -> API Tokens)",
    style=style_label, layout=widgets.Layout(width="75%")
)
btn_fetch_projects = widgets.Button(
    description="🔄 Fetch My Projects",
    button_style="info",
    tooltip="Connect to Supervisely and list all available projects across your teams",
    layout=widgets.Layout(width="22%", height="36px")
)
token_row = widgets.HBox([token_input, btn_fetch_projects], layout=full_layout)

project_dropdown = widgets.Dropdown(
    options=[("⚠️ Please enter your API token and click 'Fetch My Projects'", None)],
    description="📁 Select Project:",
    tooltip="Choose which annotated Supervisely project to download and train on",
    style=style_label, layout=full_layout
)

project_status_html = widgets.HTML(
    value="<div style='color:#64748b; font-size:13px; padding:6px 4px;'>💡 <i>Enter your API token above and click 'Fetch My Projects' to browse all your projects.</i></div>"
)

api_state = {"api": None, "current_images_count": 0}

fetch_busy = {"busy": False}

def fetch_projects(_=None):
    if fetch_busy["busy"]:
        return
    token = token_input.value.strip()
    server = server_input.value.strip()
    if not token:
        project_status_html.value = "<div style='color:#dc2626; font-weight:bold; padding:4px;'>❌ Please enter your Supervisely API token!</div>"
        return
    fetch_busy["busy"] = True
    project_status_html.value = "<div style='color:#2563eb; font-weight:600; padding:4px;'>⏳ Connecting to Supervisely and fetching team projects...</div>"
    btn_fetch_projects.disabled = True
    try:
        api = sly.Api(server_address=server, token=token)
        api_state["api"] = api

        all_projects = []
        teams = api.team.get_list()
        for team in teams:
            p_list = api.project.get_list(team_id=team.id)
            all_projects.extend(p_list)

        if not all_projects:
            project_status_html.value = "<div style='color:#d97706; font-weight:bold; padding:4px;'>⚠️ Connected, but no projects were found in your teams.</div>"
            project_dropdown.options = [("No projects found", None)]
            return

        dropdown_options = [
            (f"{p.name} (ID: {p.id})", p.id) for p in all_projects
        ]
        project_dropdown.options = dropdown_options
        project_dropdown.value = dropdown_options[0][1]
        update_project_details()
    except Exception as e:
        project_status_html.value = f"<div style='color:#dc2626; font-weight:bold; padding:4px;'>❌ Connection Error: {e}</div>"
    finally:
        fetch_busy["busy"] = False
        btn_fetch_projects.disabled = False

def update_project_details(change=None):
    pid = project_dropdown.value
    api = api_state.get("api")
    if not pid or not api:
        return
    try:
        p_info = api.project.get_info_by_id(pid)
        meta_json = api.project.get_meta(pid)
        meta_obj = sly.ProjectMeta.from_json(meta_json)
        classes = [c.name for c in meta_obj.obj_classes]
        datasets = api.dataset.get_list(pid)

        dataset_sum = sum([getattr(d, "items_count", None) or getattr(d, "images_count", None) or 0 for d in datasets])
        total_items = getattr(p_info, "items_count", None) or getattr(p_info, "images_count", None) or dataset_sum
        api_state["current_images_count"] = total_items

        # Automatic detection of dedicated validation dataset
        val_ds = next((d for d in datasets if d.name.strip().lower() in ["validation", "val"]), None)
        if val_ds:
            api_state["has_val_dataset"] = True
            api_state["val_dataset_name"] = val_ds.name
            val_items_cnt = getattr(val_ds, "items_count", None) or getattr(val_ds, "images_count", None) or 0
            api_state["val_dataset_items"] = val_items_cnt
            val_split_slider.disabled = True
            val_dataset_notice_html.value = f"""
            <div style='background:#ecfdf5; border:1px solid #6ee7b7; color:#065f46; padding:8px 12px; border-radius:6px; font-size:12px; margin-top:6px; display:flex; align-items:center; gap:8px;'>
                <span style='font-size:16px;'>🎯</span>
                <div>
                    <b>validation dataset detected:</b> <code>{val_ds.name}</code> ({val_items_cnt} images).<br>
                    <span style='color:#047857; font-size:11px;'>Validation split slider is locked; this dataset will automatically be used for evaluation.</span>
                </div>
            </div>
            """
        else:
            api_state["has_val_dataset"] = False
            api_state["val_dataset_name"] = None
            api_state["val_dataset_items"] = 0
            val_split_slider.disabled = False
            val_dataset_notice_html.value = ""

        class_pills = "".join([
            f"<span style='background:#e0f2fe; color:#0369a1; padding:2px 8px; border-radius:12px; font-size:11px; font-weight:600; margin:2px 3px 2px 0; display:inline-block;'>{c}</span>"
            for c in classes
        ]) or "<span style='color:#94a3b8; font-style:italic;'>No classes defined yet!</span>"

        val_ds_badge = ""
        if val_ds:
            val_ds_badge = f" &nbsp;|&nbsp; <span style='background:#ecfdf5; color:#065f46; padding:2px 8px; border-radius:10px; font-size:11px; font-weight:700;'>🎯 Val Dataset: {val_ds.name}</span>"

        project_status_html.value = f"""
        <div style='background:#f8fafc; border:1px solid #cbd5e1; padding:12px 14px; border-radius:8px; font-size:13px; margin-top:4px;'>
            <div style='display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;'>
                <span style='font-weight:700; color:#0f172a; font-size:14px;'>📁 {p_info.name}</span>
                <span style='background:#10b981; color:white; padding:2px 8px; border-radius:10px; font-size:11px; font-weight:700;'>Project ID: {p_info.id}</span>
            </div>
            <div style='color:#475569; font-size:12px; margin-bottom:8px;'>
                📂 <b>Datasets:</b> {len(datasets)}{val_ds_badge} &nbsp;|&nbsp; 🖼️ <b>Total Images:</b> {total_items} &nbsp;|&nbsp; 🏷️ <b>Type:</b> {p_info.type}
            </div>
            <div>
                <b style='color:#334155; font-size:12px;'>Classes ({len(classes)}):</b><br>
                <div style='margin-top:4px;'>{class_pills}</div>
            </div>
        </div>
        """
        update_training_calc()
    except Exception as e:
        project_status_html.value = f"<div style='color:#dc2626; padding:4px;'>Could not inspect project metadata: {e}</div>"

btn_fetch_projects.on_click(fetch_projects)
project_dropdown.observe(update_project_details, names="value")

# Section 2: Model Selection
family_dropdown = widgets.Dropdown(
    options=list(MODEL_CATALOG.keys()),
    value="RT-DETRv2 (Hugging Face Transformers)",
    description="🧠 Model Family:",
    tooltip="Select object detection architecture: RT-DETRv2, DEIMv2, MMDetection, YOLOX, YOLO11, YOLOv10, YOLOv9, YOLOv8, YOLOv5u, or YOLO-World",
    style=style_label, layout=full_layout
)
checkpoint_dropdown = widgets.Dropdown(
    options=MODEL_CATALOG["RT-DETRv2 (Hugging Face Transformers)"]["variants"],
    value="PekingU/rtdetr_v2_r18vd",
    description="📦 Model Variant:",
    tooltip="Pretrained model scale (Nano/Small for high FPS, Medium/Large/X for maximum precision)",
    style=style_label, layout=full_layout
)
def build_model_badge_html(family_name):
    cat = MODEL_CATALOG[family_name]
    return f"""
    <div style='{cat["badge_style"]} padding:10px 14px; border-radius:8px; font-size:12px; margin-top:8px;'>
        <div style='display:flex; justify-content:space-between; align-items:center; margin-bottom:6px; flex-wrap:wrap; gap:6px;'>
            <span style='font-weight:700; font-size:13px;'>{family_name}</span>
            {cat["license_pill"]}
        </div>
        <div style='line-height:1.4;'>{cat["badge"]}</div>
    </div>
    """

model_info_html = widgets.HTML(value=build_model_badge_html("RT-DETRv2 (Hugging Face Transformers)"))

def get_gpu_vram_gb():
    if torch.cuda.is_available():
        try:
            return torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
        except Exception:
            return 15.0
    return 0.0


def resolve_optimal_batch_size(family_name, ckpt_name, imgsz, train_imgs_count=None):
    """Automatically determine the optimal batch size based on GPU VRAM, model scale, image size, and dataset size."""
    vram_gb = get_gpu_vram_gb()
    if vram_gb <= 0:
        return 4

    c = (ckpt_name or "").lower()
    f = (family_name or "").lower()
    is_transformer = any(k in f for k in ("rt-detr", "deimv2", "world"))

    # Base batch size calibrated for a 15 GB T4 GPU at 640x640
    if any(k in c for k in ("atto", "femto", "pico", "nano", "tiny", "11n", "10n", "9t", "8n", "5nu", "_n_")):
        base_bs = 16 if is_transformer else 32
    elif any(k in c for k in ("r18vd", "r34vd", "11s", "10s", "9s", "8s", "5su", "_s.", "_s_", "s-world")):
        base_bs = 8 if is_transformer else 16
    elif any(k in c for k in ("r50vd", "11m", "10m", "10b", "9m", "9c", "8m", "5mu", "_m.", "_m_", "m-world", "faster-rcnn")):
        base_bs = 8
    elif any(k in c for k in ("11l", "10l", "8l", "5lu", "_l.", "_l_", "l-world")):
        base_bs = 4 if is_transformer else 8
    else:
        # Extra-large / r101vd / 9e / _x
        base_bs = 4

    vram_factor = max(0.5, vram_gb / 15.0)
    res_factor = (640.0 / max(320.0, float(imgsz))) ** 2
    raw_target = base_bs * vram_factor * res_factor

    # Snap to nearest safe power-of-two batch size
    valid_batches = [2, 4, 8, 16, 32, 64]
    chosen = 2
    for b in valid_batches:
        if b <= raw_target * 1.15:
            chosen = b

    # Cap by training dataset size so we have meaningful gradient steps per epoch
    if train_imgs_count and train_imgs_count >= 2:
        max_sensible = max(2, int(2 ** math.floor(math.log2(max(2, train_imgs_count)))))
        chosen = min(chosen, max_sensible)

    return max(2, min(64, chosen))


def on_family_change(change):
    if change["name"] == "value":
        selected_family = change["new"]
        new_variants = MODEL_CATALOG[selected_family]["variants"]
        checkpoint_dropdown.options = new_variants
        checkpoint_dropdown.value = new_variants[0][1]
        model_info_html.value = build_model_badge_html(selected_family)
        update_training_calc()
family_dropdown.observe(on_family_change)

# Section 3: Training Parameters & Export Targets
epochs_slider = widgets.IntSlider(
    value=30, min=1, max=200, step=1,
    description="🔄 Epochs:",
    tooltip="Number of complete training iterations over the entire dataset (10-30 for quick fine-tuning, 50-100+ for high accuracy)",
    style=style_label, layout=full_layout
)
batch_dropdown = widgets.Dropdown(
    options=[
        ("⚡ Auto (GPU VRAM & Model Optimized)", -1),
        ("2 (Ultra-Low VRAM)", 2),
        ("4 (Conservative)", 4),
        ("8 (Standard)", 8),
        ("16 (High Throughput)", 16),
        ("32 (Large GPU / Nano Models)", 32),
        ("64 (Max Throughput / A100)", 64),
    ],
    value=-1,
    description="📚 Batch Size:",
    tooltip="Select 'Auto' to automatically pick the optimal batch size for your GPU VRAM, model variant, and image resolution",
    style=style_label, layout=widgets.Layout(width="380px")
)
imgsz_dropdown = widgets.Dropdown(
    options=[("480x480 (Fast)", 480), ("640x640 (Standard)", 640), ("800x800 (High-Res)", 800)],
    value=640,
    description="🖼️ Image Size:",
    tooltip="Input resolution in pixels (640 is standard; 800 improves tiny object recall at higher VRAM cost)",
    style=style_label, layout=widgets.Layout(width="320px")
)
lr_dropdown = widgets.Dropdown(
    options=[
        ("0.001 (1e-3 | Fast CNN / YOLO)", 1e-3),
        ("0.0005 (5e-4 | Balanced)", 5e-4),
        ("0.0001 (1e-4 | Recommended Default)", 1e-4),
        ("0.00005 (5e-5 | Conservative Fine-Tuning)", 5e-5),
    ],
    value=1e-4,
    description="⚡ Learning Rate:",
    tooltip="Initial optimizer learning rate (1e-4 is optimal for Transformers/DETR; 5e-4 or 1e-3 works well for YOLO/CNN models)",
    style=style_label, layout=widgets.Layout(width="440px")
)
val_split_slider = widgets.FloatSlider(
    value=0.2, min=0.05, max=0.4, step=0.05,
    readout_format=".0%",
    description="✂️ Validation Split:",
    tooltip="Fraction of dataset reserved for validation evaluation (e.g. 0.20 = 20% validation, 80% training)",
    style=style_label, layout=full_layout
)
val_dataset_notice_html = widgets.HTML(value="")

# Export Format Checkboxes
chk_export_pytorch = widgets.Checkbox(
    value=True,
    description="💾 PyTorch (.pt / .pth / safetensors)",
    disabled=True,
    tooltip="Native PyTorch weights (always included)",
    layout=widgets.Layout(width="230px")
)
chk_export_onnx = widgets.Checkbox(
    value=True,
    description="🌐 ONNX (.onnx)",
    tooltip="Universal cross-platform format for Windows, Linux, TensorRT, and Web",
    layout=widgets.Layout(width="180px")
)
chk_export_coreml = widgets.Checkbox(
    value=True,
    description="📱 Apple CoreML (.mlpackage)",
    tooltip="Optimized for Apple Neural Engine on iOS 16+, iPadOS, and macOS",
    layout=widgets.Layout(width="250px")
)
chk_export_litert = widgets.Checkbox(
    value=True,
    description="🤖 Google LiteRT / TFLite (.tflite)",
    tooltip="Google LiteRT on-device runtime for Android, Coral, Raspberry Pi, and Mobile",
    layout=widgets.Layout(width="270px")
)

export_formats_row = widgets.HBox(
    [chk_export_pytorch, chk_export_onnx, chk_export_coreml, chk_export_litert],
    layout=widgets.Layout(flex_flow="wrap", gap="8px", margin="4px 0 8px 0")
)

training_calc_html = widgets.HTML(value="")

def update_training_calc(*args):
    total_imgs = api_state.get("current_images_count") or 122
    val_ratio = val_split_slider.value
    raw_batch = batch_dropdown.value
    epochs = epochs_slider.value

    has_val_ds = api_state.get("has_val_dataset", False)
    val_ds_name = api_state.get("val_dataset_name", "validation")
    val_items = api_state.get("val_dataset_items", 0)

    if has_val_ds and val_items > 0:
        val_imgs = min(val_items, max(1, total_imgs - 1)) if total_imgs > 1 else val_items
        train_imgs = max(1, total_imgs - val_imgs)
        val_pct_str = f"{val_imgs / total_imgs:.0%}" if total_imgs > 0 else "--"
        split_desc = f"~<b>{train_imgs}</b> Train images &nbsp;|&nbsp; <b>{val_imgs}</b> Val images ({val_pct_str}, from dedicated '<code>{val_ds_name}</code>' dataset)"
    else:
        val_imgs = max(1, int(total_imgs * val_ratio))
        train_imgs = max(1, total_imgs - val_imgs)
        split_desc = f"~<b>{train_imgs}</b> Train images ({(1-val_ratio):.0%}) &nbsp;|&nbsp; ~<b>{val_imgs}</b> Val images ({val_ratio:.0%})"

    if raw_batch == -1:
        batch_sz = resolve_optimal_batch_size(
            family_dropdown.value, checkpoint_dropdown.value, imgsz_dropdown.value, train_imgs
        )
        vram_gb = get_gpu_vram_gb()
        vram_tag = f"{vram_gb:.1f} GB VRAM" if vram_gb > 0 else "CPU"
        batch_desc = f"⚡ <b>Auto-Batch: {batch_sz}</b> <span style='color:#0284c7; font-size:11px;'>({vram_tag}, {imgsz_dropdown.value}px)</span> &nbsp;|&nbsp; "
    else:
        batch_sz = raw_batch
        batch_desc = f"📚 <b>Batch: {batch_sz}</b> &nbsp;|&nbsp; "

    steps_per_epoch = max(1, math.ceil(train_imgs / batch_sz))
    total_steps = steps_per_epoch * epochs

    training_calc_html.value = f"""
    <div style='background:#f1f5f9; border:1px solid #cbd5e1; padding:8px 12px; border-radius:6px; font-size:12px; color:#334155; margin-top:8px;'>
        📊 <b>Training Calculation:</b> {split_desc} &nbsp;|&nbsp; 
        {batch_desc}<b>{steps_per_epoch}</b> steps/epoch &nbsp;|&nbsp; 
        <b>{total_steps:,}</b> total optimizer steps 
        <span style='color:#64748b; font-size:11px;'>(each step increases the epoch counter by ~{1.0/steps_per_epoch:.2f})</span>
    </div>
    """

epochs_slider.observe(update_training_calc, names="value")
batch_dropdown.observe(update_training_calc, names="value")
checkpoint_dropdown.observe(update_training_calc, names="value")
imgsz_dropdown.observe(update_training_calc, names="value")
val_split_slider.observe(update_training_calc, names="value")
update_training_calc()

# Section 4: Actions, Live Dashboard & Console
btn_train = widgets.Button(
    description="🚀 Start Training",
    button_style="success",
    tooltip="Download dataset, launch training on GPU, and export to chosen formats",
    layout=widgets.Layout(width="220px", height="42px")
)
btn_stop = widgets.Button(
    description="⏹️ Stop Training",
    button_style="danger",
    disabled=True,
    tooltip="Safely stop training after current step and save the best model checkpoint so far",
    layout=widgets.Layout(width="180px", height="42px")
)
btn_download_zip = widgets.Button(
    description="💾 Download Model Bundle (.zip)",
    button_style="primary",
    disabled=True,
    tooltip="Compress all trained weights and exported models (CoreML, LiteRT, ONNX, PyTorch) into a .zip file and download",
    layout=widgets.Layout(width="280px", height="42px")
)

progress_bar = widgets.IntProgress(
    value=0, min=0, max=100,
    description="Progress:",
    bar_style="info",
    tooltip="Overall pipeline progress (Download -> Prep -> Train -> Export -> Package)",
    layout=widgets.Layout(width="98%")
)
status_text = widgets.HTML(value="<div style='background:#f8fafc; border:1px solid #cbd5e1; color:#0f172a; padding:8px 12px; border-radius:6px; font-size:12px; font-weight:600; margin-top:4px;'>✅ Ready to start. Select your project and click <b>Start Training</b>.</div>")

# View Selector: Visual Dashboard vs Raw Console
view_toggle = widgets.ToggleButtons(
    options=[("📊 Visual Dashboard (Clean UI)", "dashboard"), ("💻 Raw Console Logs", "console")],
    value="dashboard",
    tooltip="Switch between clean real-time visual dashboard and raw terminal logs",
    style={"button_width": "220px"},
    layout=widgets.Layout(margin="10px 0 10px 0")
)

# Helper: Time Formatter
def format_seconds(sec):
    if sec is None or sec < 0:
        return "--"
    s = int(sec)
    h = s // 3600
    m = (s % 3600) // 60
    rem_s = s % 60
    if h > 0:
        return f"{h}h {m:02d}m {rem_s:02d}s"
    elif m > 0:
        return f"{m}m {rem_s:02d}s"
    else:
        return f"{rem_s}s"

# Live Dashboard State
dashboard_state = {
    "phase": "Ready",
    "epoch": 0,
    "max_epochs": 30,
    "step": 0,
    "max_steps": 0,
    "train_loss": None,
    "val_loss": None,
    "start_time": None,
    "elapsed_str": "--",
    "eta_str": "--",
    "speed_str": "--",
    "history": []
}

def generate_loss_svg(history, max_epochs=30):
    valid_train = [(h["epoch"], h["train_loss"]) for h in history if h.get("train_loss") is not None]
    valid_val = [(h["epoch"], h["val_loss"]) for h in history if h.get("val_loss") is not None]

    if not valid_train and not valid_val:
        return """
        <details style="background:#ffffff; border:1px solid #e2e8f0; border-radius:8px; margin-bottom:12px; overflow:hidden;">
            <summary style="list-style:none; padding:8px 12px; font-weight:700; color:#334155; font-size:12px; cursor:pointer; display:flex; justify-content:space-between; align-items:center; user-select:none; background:#f8fafc; outline:none;">
                <span style="display:inline-flex; align-items:center; gap:6px;">
                    <span>📉</span>
                    <span>Loss Convergence Graph</span>
                    <span style="font-size:10px; font-weight:600; color:#64748b; background:#f1f5f9; padding:1px 6px; border-radius:4px; margin-left:4px;">Drill-down</span>
                </span>
                <span style="font-size:11px; color:#64748b; font-weight:normal;">▼ Click to expand / collapse</span>
            </summary>
            <div style="background:#f8fafc; border-top:1px solid #e2e8f0; padding:18px 16px; text-align:center; color:#94a3b8; font-size:12px;">
                📈 <b>Loss convergence graph will appear here automatically</b> once the first epoch completes.
            </div>
        </details>
        """

    all_epochs = [e for e, _ in valid_train + valid_val]
    all_losses = [l for _, l in valid_train + valid_val]

    min_epoch = 1
    max_epoch = max(max_epochs or 1, max(all_epochs) if all_epochs else 1)
    if max_epoch <= 1:
        max_epoch = 2

    min_loss = min(all_losses) if all_losses else 0.0
    max_loss = max(all_losses) if all_losses else 1.0
    if min_loss == max_loss:
        max_loss += 1.0

    y_pad = (max_loss - min_loss) * 0.12
    y_min = max(0.0, min_loss - y_pad)
    y_max = max_loss + y_pad

    w = 640
    h = 135
    pad_l = 40
    pad_r = 16
    pad_t = 12
    pad_b = 22
    plot_w = w - pad_l - pad_r
    plot_h = h - pad_t - pad_b

    def get_x(ep):
        if max_epoch == min_epoch:
            return pad_l + plot_w / 2
        return pad_l + ((ep - min_epoch) / (max_epoch - min_epoch)) * plot_w

    def get_y(loss):
        if y_max == y_min:
            return pad_t + plot_h / 2
        return pad_t + plot_h - ((loss - y_min) / (y_max - y_min)) * plot_h

    # Gridlines (4 horizontal lines)
    grid_svg = ""
    for i in range(4):
        val = y_min + (y_max - y_min) * (i / 3.0)
        y_pos = pad_t + plot_h - (i / 3.0) * plot_h
        grid_svg += f'<line x1="{pad_l}" y1="{y_pos:.1f}" x2="{w - pad_r}" y2="{y_pos:.1f}" stroke="#f1f5f9" stroke-dasharray="3,3" />'
        grid_svg += f'<text x="{pad_l - 6}" y="{y_pos + 3:.1f}" fill="#94a3b8" font-size="9" text-anchor="end" font-family="sans-serif">{val:.2f}</text>'

    # X axis
    x_axis_svg = f'<line x1="{pad_l}" y1="{pad_t + plot_h}" x2="{w - pad_r}" y2="{pad_t + plot_h}" stroke="#cbd5e1" stroke-width="1" />'
    num_ticks = min(8, max_epoch)
    step_x = max(1, max_epoch // num_ticks)
    for ep in range(1, max_epoch + 1, step_x):
        x_pos = get_x(ep)
        x_axis_svg += f'<line x1="{x_pos:.1f}" y1="{pad_t + plot_h}" x2="{x_pos:.1f}" y2="{pad_t + plot_h + 3}" stroke="#94a3b8" />'
        x_axis_svg += f'<text x="{x_pos:.1f}" y="{pad_t + plot_h + 13}" fill="#94a3b8" font-size="9" text-anchor="middle" font-family="sans-serif">Ep {ep}</text>'

    # Train Loss Line
    train_pts = []
    train_dots = ""
    for ep, l in valid_train:
        cx, cy = get_x(ep), get_y(l)
        train_pts.append(f"{cx:.1f},{cy:.1f}")
        train_dots += f'<circle cx="{cx:.1f}" cy="{cy:.1f}" r="2.5" fill="#2563eb" />'
    train_line = f'<polyline points="{" ".join(train_pts)}" fill="none" stroke="#2563eb" stroke-width="2" stroke-linecap="round" stroke-linejoin="round" />' if train_pts else ""

    # Val Loss Line
    val_pts = []
    val_dots = ""
    for ep, l in valid_val:
        cx, cy = get_x(ep), get_y(l)
        val_pts.append(f"{cx:.1f},{cy:.1f}")
        val_dots += f'<circle cx="{cx:.1f}" cy="{cy:.1f}" r="2.5" fill="#10b981" />'
    val_line = f'<polyline points="{" ".join(val_pts)}" fill="none" stroke="#10b981" stroke-width="2" stroke-linecap="round" stroke-linejoin="round" />' if val_pts else ""

    svg_content = f"""
    <details open style="background:#ffffff; border:1px solid #e2e8f0; border-radius:8px; margin-bottom:12px; overflow:hidden;">
        <summary style="list-style:none; padding:8px 12px; font-weight:700; color:#334155; font-size:12px; cursor:pointer; display:flex; justify-content:space-between; align-items:center; user-select:none; background:#f8fafc; outline:none; border-bottom:1px solid #e2e8f0;">
            <span style="display:inline-flex; align-items:center; gap:6px;">
                <span>📉</span>
                <span>Loss Convergence Graph</span>
                <span style="font-size:10px; font-weight:600; color:#0284c7; background:#e0f2fe; padding:1px 6px; border-radius:4px; margin-left:4px;">Drill-down</span>
            </span>
            <div style="display:inline-flex; align-items:center; gap:12px; font-size:11px;">
                <span style="color:#2563eb; display:inline-flex; align-items:center; gap:4px;"><span style="width:10px; height:3px; background:#2563eb; display:inline-block; border-radius:2px;"></span> Train</span>
                <span style="color:#10b981; display:inline-flex; align-items:center; gap:4px;"><span style="width:10px; height:3px; background:#10b981; display:inline-block; border-radius:2px;"></span> Val</span>
                <span style="color:#64748b; font-weight:normal; font-size:11px; margin-left:4px;">▼ Click to expand / collapse</span>
            </div>
        </summary>
        <div style="padding:8px 12px 6px 12px; background:#ffffff;">
            <svg viewBox="0 0 {w} {h}" style="width:100%; max-height:135px; height:auto; display:block;">
                {grid_svg}
                {x_axis_svg}
                {train_line}
                {val_line}
                {train_dots}
                {val_dots}
            </svg>
        </div>
    </details>
    """
    return svg_content


def render_dashboard_html():
    phase = dashboard_state["phase"]
    epoch = dashboard_state["epoch"]
    max_epochs = dashboard_state["max_epochs"]
    step = dashboard_state["step"]
    max_steps = dashboard_state["max_steps"]
    train_loss = dashboard_state["train_loss"]
    val_loss = dashboard_state["val_loss"]
    history = dashboard_state["history"]
    eta_str = dashboard_state["eta_str"]
    elapsed_str = dashboard_state["elapsed_str"]
    speed_str = dashboard_state["speed_str"]

    epoch_str = f"{epoch} / {max_epochs}" if max_epochs else f"{epoch}"
    step_str = f"Step {step} / {max_steps}" if max_steps else (f"Step {step}" if step else "Waiting...")
    t_loss_str = f"{train_loss:.4f}" if (train_loss is not None and isinstance(train_loss, (int, float))) else "--"
    v_loss_str = f"{val_loss:.4f}" if (val_loss is not None and isinstance(val_loss, (int, float))) else "--"

    # Status badge color
    phase_bg = "#e0f2fe"
    phase_color = "#0284c7"
    if "Training" in phase:
        phase_bg = "#fef3c7"
        phase_color = "#d97706"
    elif "Completed" in phase or "Ready" in phase:
        phase_bg = "#dcfce7"
        phase_color = "#15803d"
    elif "Stopped" in phase or "Error" in phase:
        phase_bg = "#fee2e2"
        phase_color = "#dc2626"

    # Loss SVG Chart
    loss_chart_html = generate_loss_svg(history, max_epochs=max_epochs)

    rows_html = ""
    for item in reversed(history[-10:]):
        ep = item.get("epoch", "-")
        tl = f"{item['train_loss']:.4f}" if item.get("train_loss") is not None else "--"
        vl = f"{item['val_loss']:.4f}" if item.get("val_loss") is not None else "--"
        st = item.get("status", "Completed")
        rows_html += f"<tr style='border-bottom:1px solid #f1f5f9; background:#ffffff;'><td style='padding:6px 10px; font-weight:600; color:#0f172a;'>Epoch {ep}</td><td style='padding:6px 10px; color:#2563eb; font-weight:600;'>{tl}</td><td style='padding:6px 10px; color:#059669; font-weight:600;'>{vl}</td><td style='padding:6px 10px;'><span style='background:#dcfce7; color:#15803d; padding:2px 6px; border-radius:4px; font-size:11px; font-weight:600;'>{st}</span></td></tr>"

    if not rows_html:
        rows_html = "<tr><td colspan='4' style='padding:14px; text-align:center; color:#94a3b8; font-style:italic;'>No epoch metrics recorded yet. Start training to view live metrics!</td></tr>"

    return f"""
    <div style="background:#ffffff; border:1px solid #e2e8f0; border-radius:10px; padding:16px; font-family:-apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; box-shadow:0 1px 3px rgba(0,0,0,0.05);">
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:12px; border-bottom:1px solid #f1f5f9; padding-bottom:8px;">
            <div style="font-weight:700; color:#1e293b; font-size:14px; display:flex; align-items:center; gap:6px;">
                <span>📊 Live Training Dashboard</span>
            </div>
            <span style="background:{phase_bg}; color:{phase_color}; padding:4px 12px; border-radius:12px; font-size:12px; font-weight:700;">{phase}</span>
        </div>

        <!-- KPI Grid with Time Remaining / ETA -->
        <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(130px, 1fr)); gap:10px; margin-bottom:14px;">
            <div style="background:#f8fafc; border:1px solid #e2e8f0; border-radius:8px; padding:12px 10px; text-align:center;">
                <div style="color:#64748b; font-size:11px; font-weight:600; text-transform:uppercase;">Current Epoch</div>
                <div style="color:#0f172a; font-size:18px; font-weight:700; margin-top:4px;">{epoch_str}</div>
                <div style="color:#64748b; font-size:11px; margin-top:2px;">{step_str}</div>
            </div>

            <div style="background:#f8fafc; border:1px solid #e2e8f0; border-radius:8px; padding:12px 10px; text-align:center;">
                <div style="color:#64748b; font-size:11px; font-weight:600; text-transform:uppercase;">Training Loss</div>
                <div style="color:#2563eb; font-size:18px; font-weight:700; margin-top:4px;">{t_loss_str}</div>
                <div style="color:#64748b; font-size:11px; margin-top:2px;">Latest optimization step</div>
            </div>

            <div style="background:#f8fafc; border:1px solid #e2e8f0; border-radius:8px; padding:12px 10px; text-align:center;">
                <div style="color:#64748b; font-size:11px; font-weight:600; text-transform:uppercase;">Val Loss / mAP50</div>
                <div style="color:#059669; font-size:18px; font-weight:700; margin-top:4px;">{v_loss_str}</div>
                <div style="color:#64748b; font-size:11px; margin-top:2px;">Holdout validation metric</div>
            </div>

            <div style="background:#f8fafc; border:1px solid #e2e8f0; border-radius:8px; padding:12px 10px; text-align:center;">
                <div style="color:#64748b; font-size:11px; font-weight:600; text-transform:uppercase;">Time Remaining (ETA)</div>
                <div style="color:#d97706; font-size:17px; font-weight:700; margin-top:4px;">{eta_str}</div>
                <div style="color:#64748b; font-size:11px; margin-top:2px;">Elapsed: {elapsed_str} &bull; {speed_str}</div>
            </div>
        </div>

        <!-- Graphical Loss Convergence Chart -->
        {loss_chart_html}

        <!-- Completed Epochs Table -->
        <div style="font-weight:600; font-size:12px; color:#475569; margin-bottom:6px;">📋 Completed Epoch Log:</div>
        <div style="max-height:150px; overflow-y:auto; border:1px solid #e2e8f0; border-radius:6px;">
            <table style="width:100%; border-collapse:collapse; font-size:12px; text-align:left;">
                <thead style="background:#f8fafc; border-bottom:1px solid #e2e8f0; color:#475569; position:sticky; top:0;">
                    <tr>
                        <th style="padding:6px 10px; color:#334155;">Epoch</th>
                        <th style="padding:6px 10px; color:#334155;">Train Loss</th>
                        <th style="padding:6px 10px; color:#334155;">Val Loss / mAP50</th>
                        <th style="padding:6px 10px; color:#334155;">Status</th>
                    </tr>
                </thead>
                <tbody>
                    {rows_html}
                </tbody>
            </table>
        </div>
    </div>
    """

dashboard_html = widgets.HTML(value=render_dashboard_html())
dashboard_container = widgets.VBox([dashboard_html], layout=widgets.Layout(width="100%"))

log_output = widgets.Output(
    layout=widgets.Layout(
        border="1px solid #334155",
        padding="12px",
        max_height="380px",
        overflow_y="auto",
        background_color="#0f172a",
        color="#f8fafc"
    )
)

# Robust container swapping: avoids Colab CSS display:none override issues
view_container = widgets.VBox([dashboard_container], layout=widgets.Layout(width="100%"))

def on_view_toggle(change=None):
    if isinstance(change, dict):
        val = change.get("new", view_toggle.value)
    elif isinstance(change, str):
        val = change
    else:
        val = view_toggle.value

    if val == "dashboard":
        view_container.children = (dashboard_container,)
    else:
        view_container.children = (log_output,)

view_toggle.observe(on_view_toggle, names="value")

stop_state = {"stop": False}

def on_stop_click(_):
    stop_state["stop"] = True
    btn_stop.disabled = True
    status_text.value = "<b style='color:#dc2626;'>⏹️ Stop requested! Gracefully finalizing current step and saving model...</b>"
    dashboard_state["phase"] = "Stopping..."
    dashboard_html.value = render_dashboard_html()

btn_stop.on_click(on_stop_click)

last_saved_model_path = {"path": None}

def update_ui_metric(phase=None, epoch=None, max_epochs=None, step=None, max_steps=None, train_loss=None, val_loss=None, append_history=False):
    if phase:
        dashboard_state["phase"] = phase
    if epoch is not None:
        dashboard_state["epoch"] = epoch
    if max_epochs is not None:
        dashboard_state["max_epochs"] = max_epochs
    if step is not None:
        dashboard_state["step"] = step
    if max_steps is not None:
        dashboard_state["max_steps"] = max_steps
    if train_loss is not None:
        dashboard_state["train_loss"] = train_loss
    if val_loss is not None:
        dashboard_state["val_loss"] = val_loss

    # Calculate Time Elapsed & Remaining ETA
    st_time = dashboard_state.get("start_time")
    if st_time is not None:
        elapsed = time.time() - st_time
        dashboard_state["elapsed_str"] = format_seconds(elapsed)
        
        # Calculate Remaining Time based on steps or epochs
        m_steps = dashboard_state.get("max_steps") or 0
        cur_step = dashboard_state.get("step") or 0
        m_epochs = dashboard_state.get("max_epochs") or 0
        cur_epoch = dashboard_state.get("epoch") or 0

        if m_steps > 0 and cur_step > 0:
            prog_ratio = min(1.0, cur_step / m_steps)
            sec_per_step = elapsed / cur_step
            rem_sec = max(0, (m_steps - cur_step) * sec_per_step)
            dashboard_state["eta_str"] = f"~{format_seconds(rem_sec)} remaining"
            dashboard_state["speed_str"] = f"{1.0/sec_per_step:.2f} it/s" if sec_per_step > 0 else "--"
        elif m_epochs > 0 and cur_epoch > 0:
            prog_ratio = min(1.0, cur_epoch / m_epochs)
            sec_per_ep = elapsed / cur_epoch
            rem_sec = max(0, (m_epochs - cur_epoch) * sec_per_ep)
            dashboard_state["eta_str"] = f"~{format_seconds(rem_sec)} remaining"
            dashboard_state["speed_str"] = f"{sec_per_ep:.1f} s/epoch"
        else:
            dashboard_state["eta_str"] = "Estimating..."
            dashboard_state["speed_str"] = "--"

    if append_history and epoch is not None:
        dashboard_state["history"].append({
            "epoch": epoch,
            "train_loss": dashboard_state["train_loss"],
            "val_loss": dashboard_state["val_loss"],
            "status": "Evaluated" if val_loss is not None else "Logged"
        })

    dashboard_html.value = render_dashboard_html()

# --- 4. LOGIC & ON-DEMAND DEPENDENCIES ---
IMPORT_NAME_MAP = {
    "pyyaml": "yaml",
    "pillow": "PIL",
    "opencv_python": "cv2",
    "scikit_learn": "sklearn",
}

def ensure_packages(pkgs):
    to_install = []
    is_mmdet = any("mmdet" in p or "mmcv" in p or "mmengine" in p for p in pkgs)
    if is_mmdet:
        try:
            import mmdet
            import mmcv
            import mmengine
        except ImportError:
            print("📦 Installing OpenMMLab dependencies via MIM (openmim, mmengine, mmcv, mmdet)...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "openmim"])
            subprocess.check_call(["mim", "install", "mmengine", "mmcv>=2.0.0", "mmdet>=3.0.0"])
            print("✅ OpenMMLab packages successfully installed!")
        pkgs = [p for p in pkgs if not any(x in p for x in ("openmim", "mmengine", "mmcv", "mmdet"))]

    for pkg in pkgs:
        raw_name = pkg.split(">=")[0].split("==")[0].split("<")[0].strip().replace("-", "_")
        pkg_import = IMPORT_NAME_MAP.get(raw_name.lower(), raw_name)
        try:
            __import__(pkg_import)
        except ImportError:
            to_install.append(pkg)
    if to_install:
        print(f"📦 Installing required packages: {', '.join(to_install)}...")
        try:
            import PIL
            p_pin = [f"pillow=={PIL.__version__}"]
        except Exception:
            p_pin = []
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + p_pin + to_install)
        print("✅ Packages successfully installed!")


def prepare_coco_dataset(target_dir, train_samples, val_samples, meta, label2id):
    """Convert Supervisely train/val splits into standard MS-COCO directory and JSON format."""
    if os.path.exists(target_dir):
        shutil.rmtree(target_dir)
    os.makedirs(f"{target_dir}/annotations", exist_ok=True)
    os.makedirs(f"{target_dir}/train2017", exist_ok=True)
    os.makedirs(f"{target_dir}/val2017", exist_ok=True)

    categories = [{"id": cid, "name": name, "supercategory": "none"} for name, cid in label2id.items()]

    for split_name, split_samples, json_file in [
        ("train2017", train_samples, "instances_train2017.json"),
        ("val2017", val_samples, "instances_val2017.json"),
    ]:
        images_list = []
        annotations_list = []
        ann_id = 1
        for sample_idx, sample in enumerate(split_samples, start=1):
            ann = sly.Annotation.load_json_file(sample["ann_path"], meta)
            ih, iw = ann.img_size
            stem = f"{sample['dataset_name']}_{Path(sample['item_name']).stem}"
            ext = Path(sample["item_name"]).suffix
            dest_filename = f"{stem}{ext}"
            shutil.copy2(sample["img_path"], f"{target_dir}/{split_name}/{dest_filename}")

            images_list.append({
                "id": sample_idx,
                "file_name": dest_filename,
                "height": ih,
                "width": iw,
            })

            for l in ann.labels:
                if l.obj_class.name not in label2id:
                    continue
                cid = label2id[l.obj_class.name]
                bb = l.geometry.to_bbox()
                w = float(bb.width)
                h = float(bb.height)
                if w > 0 and h > 0:
                    annotations_list.append({
                        "id": ann_id,
                        "image_id": sample_idx,
                        "category_id": cid,
                        "bbox": [float(bb.left), float(bb.top), w, h],
                        "area": w * h,
                        "iscrowd": 0,
                    })
                    ann_id += 1

        coco_dict = {
            "images": images_list,
            "annotations": annotations_list,
            "categories": categories,
        }
        with open(f"{target_dir}/annotations/{json_file}", "w", encoding="utf-8") as f:
            json.dump(coco_dict, f)


pipeline_busy = {"busy": False}

def run_training_pipeline(_=None):
    if pipeline_busy["busy"]:
        return
    pipeline_busy["busy"] = True
    log_output.clear_output()
    btn_train.disabled = True
    btn_stop.disabled = False
    btn_download_zip.disabled = True
    stop_state["stop"] = False
    progress_bar.bar_style = "info"
    progress_bar.value = 5

    dashboard_state["history"] = []
    dashboard_state["train_loss"] = None
    dashboard_state["val_loss"] = None
    dashboard_state["step"] = 0
    dashboard_state["epoch"] = 0
    dashboard_state["start_time"] = None
    dashboard_state["elapsed_str"] = "--"
    dashboard_state["eta_str"] = "Starting..."
    dashboard_state["speed_str"] = "--"

    pid = project_dropdown.value
    token = token_input.value.strip()
    server = server_input.value.strip()
    family = family_dropdown.value
    checkpoint = checkpoint_dropdown.value
    epochs = epochs_slider.value
    batch_size = batch_dropdown.value
    imgsz = imgsz_dropdown.value
    lr = lr_dropdown.value
    val_split = val_split_slider.value

    export_onnx = chk_export_onnx.value
    export_coreml = chk_export_coreml.value
    export_litert = chk_export_litert.value

    export_results = {"success": [], "warnings": []}

    update_ui_metric(phase="Preparing Environment", max_epochs=epochs)

    with log_output:
        try:
            if not token:
                raise ValueError("Please enter your Supervisely API token!")
            if not pid:
                raise ValueError("Please select a project from the dropdown menu!")

            # 1. Ensure required dependencies for selected model
            status_text.value = f"<b>⏳ Checking dependencies for {family}...</b>"
            ensure_packages(MODEL_CATALOG[family]["packages"])
            progress_bar.value = 15

            # 2. Reliable Project Download via Project.download
            status_text.value = "<b>⏳ Downloading project from Supervisely...</b>"
            update_ui_metric(phase="Downloading Project")
            api = api_state.get("api") or sly.Api(server_address=server, token=token)
            p_info = api.project.get_info_by_id(pid)
            if p_info is None:
                raise ValueError(f"Project {pid} not found or access denied.")

            raw_dir = "/content/sly_project"
            if os.path.exists(raw_dir):
                shutil.rmtree(raw_dir)

            datasets = api.dataset.get_list(pid)
            expected_count = getattr(p_info, "items_count", None) or getattr(p_info, "images_count", None) or sum([getattr(d, "items_count", 0) or getattr(d, "images_count", 0) for d in datasets])
            print(f"📥 Downloading project '{p_info.name}' ({expected_count} images) in batches of 50...")
            sly.Project.download(
                api=api,
                project_id=pid,
                dest_dir=raw_dir,
                batch_size=50,
                log_progress=False,
                save_images=True
            )
            print(f"✅ Successfully downloaded {expected_count} images!")
            progress_bar.value = 35

            # 3. Prepare Dataset & Train/Val Split
            status_text.value = "<b>⏳ Preparing Train/Val dataset split...</b>"
            update_ui_metric(phase="Preparing Dataset")
            project_fs = sly.Project(raw_dir, sly.OpenMode.READ)
            meta = project_fs.meta
            class_names = [obj_class.name for obj_class in meta.obj_classes]
            if not class_names:
                raise ValueError("Supervisely project contains no object classes!")

            id2label = {idx: name for idx, name in enumerate(class_names)}
            label2id = {name: idx for idx, name in enumerate(class_names)}

            all_samples = []
            for ds in project_fs:
                for item_name in ds:
                    all_samples.append({
                        "dataset_name": ds.name,
                        "item_name": item_name,
                        "img_path": ds.get_item_path(item_name),
                        "ann_path": ds.get_ann_path(item_name),
                    })

            if len(all_samples) < 2:
                raise ValueError("At least 2 annotated images are required to train and validate!")

            # Check for dedicated validation dataset
            val_ds_names = [ds.name for ds in project_fs if ds.name.strip().lower() in ["validation", "val"]]

            if val_ds_names:
                chosen_val_name = val_ds_names[0]
                val_samples = [s for s in all_samples if s["dataset_name"] in val_ds_names]
                train_samples = [s for s in all_samples if s["dataset_name"] not in val_ds_names]

                if not train_samples:
                    print(f"⚠️ Notice: All images were inside validation dataset '{chosen_val_name}'. Performing fallback 80/20 split.")
                    random.seed(42)
                    random.shuffle(all_samples)
                    val_count = max(1, int(len(all_samples) * 0.2))
                    val_samples = all_samples[:val_count]
                    train_samples = all_samples[val_count:]
                elif not val_samples:
                    print(f"⚠️ Notice: Validation dataset '{chosen_val_name}' was empty. Performing fallback 80/20 split.")
                    random.seed(42)
                    random.shuffle(all_samples)
                    val_count = max(1, int(len(all_samples) * 0.2))
                    val_samples = all_samples[:val_count]
                    train_samples = all_samples[val_count:]
                else:
                    print(f"🎯 Dedicated validation dataset detected: '{chosen_val_name}' ({len(val_samples)} images)")
                    print(f"📊 Dataset split (using dedicated '{chosen_val_name}'): {len(train_samples)} Train | {len(val_samples)} Validation")
            else:
                random.seed(42)
                random.shuffle(all_samples)
                val_count = max(1, int(len(all_samples) * val_split))
                val_samples = all_samples[:val_count]
                train_samples = all_samples[val_count:]
                print(f"📊 Dataset split (random split {val_split:.0%}): {len(train_samples)} Train | {len(val_samples)} Validation")
            print(f"🏷️ Classes ({len(class_names)}): {id2label}")
            if batch_size == -1:
                batch_size = resolve_optimal_batch_size(family, checkpoint, imgsz, len(train_samples))
                vram_gb = get_gpu_vram_gb()
                print(f"⚡ Auto-Batch Mode: Selected optimal batch_size={batch_size} (GPU VRAM: {vram_gb:.1f} GB, Resolution: {imgsz}x{imgsz}, Train Samples: {len(train_samples)})")
            progress_bar.value = 45

            # 4. Model Training
            status_text.value = f"<b>🚀 Training {family} ({checkpoint})...</b>"
            dashboard_state["start_time"] = time.time()
            update_ui_metric(phase=f"Training {family}")

            if "RT-DETRv2" in family:
                from torch.utils.data import Dataset
                from transformers import RTDetrImageProcessor, RTDetrV2ForObjectDetection, TrainingArguments, Trainer, TrainerCallback

                class SLYDataset(Dataset):
                    def __init__(self, samples, proc):
                        self.samples = samples
                        self.proc = proc
                    def __len__(self):
                        return len(self.samples)
                    def __getitem__(self, idx):
                        s = self.samples[idx]
                        img = Image.open(s["img_path"]).convert("RGB")
                        ann = sly.Annotation.load_json_file(s["ann_path"], meta)
                        anns = []
                        for l in ann.labels:
                            if l.obj_class.name not in label2id:
                                continue
                            bb = l.geometry.to_bbox()
                            w, h = float(bb.width), float(bb.height)
                            if w > 0 and h > 0:
                                anns.append({
                                    "image_id": idx,
                                    "category_id": label2id[l.obj_class.name],
                                    "iscrowd": 0,
                                    "area": w * h,
                                    "bbox": [float(bb.left), float(bb.top), w, h],
                                })
                        target = {"image_id": idx, "annotations": anns}
                        enc = self.proc(images=img, annotations=target, return_tensors="pt")
                        return {"pixel_values": enc["pixel_values"].squeeze(0), "labels": enc["labels"][0]}

                proc = RTDetrImageProcessor.from_pretrained(checkpoint, size={"height": imgsz, "width": imgsz})
                train_ds = SLYDataset(train_samples, proc)
                val_ds = SLYDataset(val_samples, proc)

                model = RTDetrV2ForObjectDetection.from_pretrained(
                    checkpoint, id2label=id2label, label2id=label2id, ignore_mismatched_sizes=True
                )

                out_dir = "/content/runs/rtdetrv2_model"
                base_training_args = dict(
                    output_dir=out_dir,
                    num_train_epochs=epochs,
                    per_device_train_batch_size=batch_size,
                    per_device_eval_batch_size=batch_size,
                    learning_rate=lr,
                    weight_decay=1e-4,
                    lr_scheduler_type="cosine",
                    warmup_steps=10,
                    max_grad_norm=0.1,
                    fp16=torch.cuda.is_available(),
                    save_strategy="epoch",
                    save_total_limit=2,
                    load_best_model_at_end=True,
                    remove_unused_columns=False,
                    eval_do_concat_batches=False,
                    logging_steps=5,
                    report_to="none",
                )
                
                try:
                    args = TrainingArguments(eval_strategy="epoch", **base_training_args)
                except TypeError:
                    args = TrainingArguments(evaluation_strategy="epoch", **base_training_args)

                # Custom Callback for Live Dashboard UI & Clean Stop
                class LiveDashboardCallback(TrainerCallback):
                    def __init__(self, stop_ref, updater):
                        self.stop_ref = stop_ref
                        self.updater = updater
                        self.last_reported_epoch = -1

                    def on_step_begin(self, args, state, control, **kwargs):
                        if self.stop_ref.get("stop"):
                            control.should_training_stop = True

                    def on_log(self, args, state, control, logs=None, **kwargs):
                        if logs:
                            ep_int = int(state.epoch) if state.epoch else 0
                            self.updater(
                                phase=f"Training (Epoch {state.epoch:.2f}/{args.num_train_epochs})",
                                epoch=round(state.epoch or 0, 2),
                                max_epochs=args.num_train_epochs,
                                step=state.global_step,
                                max_steps=state.max_steps,
                                train_loss=logs.get("loss"),
                                val_loss=logs.get("eval_loss")
                            )
                            if state.max_steps:
                                progress_bar.value = int(45 + 40 * (state.global_step / state.max_steps))

                    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
                        if metrics:
                            ep_int = int(state.epoch) if state.epoch else 0
                            self.updater(
                                epoch=ep_int,
                                val_loss=metrics.get("eval_loss"),
                                append_history=True
                            )

                dashboard_cb = LiveDashboardCallback(stop_state, update_ui_metric)

                trainer = Trainer(
                    model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                    data_collator=lambda b: {
                        "pixel_values": torch.stack([x["pixel_values"] for x in b]),
                        "labels": [x["labels"] for x in b]
                    },
                    processing_class=proc,
                    callbacks=[dashboard_cb]
                )
                trainer.train()
                best_dir = f"{out_dir}/best_model"
                model.save_pretrained(best_dir)
                proc.save_pretrained(best_dir)
                last_saved_model_path["path"] = best_dir
                export_results["success"].append("PyTorch (safetensors)")

                # 5. Automated Multi-Format Export for RT-DETRv2
                progress_bar.value = 85
                class RTDetrExportWrapper(torch.nn.Module):
                    def __init__(self, m):
                        super().__init__()
                        self.m = m
                    def forward(self, x):
                        out = self.m(pixel_values=x)
                        return out.logits, out.pred_boxes

                export_wrapper = RTDetrExportWrapper(model).eval().cpu()
                dummy_input = torch.randn(1, 3, imgsz, imgsz)

                if export_onnx and not stop_state["stop"]:
                    status_text.value = "<b>⏳ Exporting RT-DETRv2 to ONNX...</b>"
                    update_ui_metric(phase="Exporting ONNX")
                    print("🌐 Exporting to ONNX format...")
                    try:
                        ensure_packages(["onnx", "onnxscript"])
                        onnx_path = os.path.join(best_dir, "model.onnx")
                        torch.onnx.export(
                            export_wrapper,
                            dummy_input,
                            onnx_path,
                            input_names=["pixel_values"],
                            output_names=["logits", "pred_boxes"],
                            dynamic_axes={"pixel_values": {0: "batch_size"}, "logits": {0: "batch_size"}, "pred_boxes": {0: "batch_size"}},
                            opset_version=17,
                            do_constant_folding=True,
                        )
                        print(f"✅ Successfully exported ONNX model: {onnx_path}")
                        export_results["success"].append("ONNX (.onnx)")
                    except Exception as e:
                        print(f"⚠️ ONNX export notice: {e}")
                        export_results["warnings"].append(f"ONNX: {e}")

                if export_coreml and not stop_state["stop"]:
                    status_text.value = "<b>⏳ Exporting RT-DETRv2 to Apple CoreML...</b>"
                    update_ui_metric(phase="Exporting CoreML")
                    print("📱 Exporting to Apple CoreML (.mlpackage)...")
                    try:
                        ensure_packages(["coremltools"])
                        import coremltools as ct
                        traced = torch.jit.trace(export_wrapper, dummy_input)
                        mlmodel = ct.convert(
                            traced,
                            inputs=[ct.TensorType(name="pixel_values", shape=(1, 3, imgsz, imgsz))],
                            minimum_deployment_target=ct.target.iOS16,
                            convert_to="mlprogram"
                        )
                        coreml_path = os.path.join(best_dir, "model.mlpackage")
                        mlmodel.save(coreml_path)
                        print(f"✅ Successfully exported Apple CoreML model: {coreml_path}")
                        export_results["success"].append("Apple CoreML (.mlpackage)")
                    except Exception as e:
                        print(f"⚠️ CoreML export notice: {e}")
                        export_results["warnings"].append(f"CoreML: {e}")

                if export_litert and not stop_state["stop"]:
                    status_text.value = "<b>⏳ Exporting RT-DETRv2 to Google LiteRT (TFLite)...</b>"
                    update_ui_metric(phase="Exporting LiteRT")
                    print("🤖 Exporting to Google LiteRT / TFLite (.tflite)...")
                    litert_success = False
                    try:
                        ensure_packages(["ai-edge-torch"])
                        import ai_edge_torch
                        edge_model = ai_edge_torch.convert(export_wrapper, (dummy_input,))
                        litert_path = os.path.join(best_dir, "model_litert.tflite")
                        edge_model.export(litert_path)
                        print(f"✅ Successfully exported Google LiteRT model: {litert_path}")
                        export_results["success"].append("Google LiteRT (.tflite)")
                        litert_success = True
                    except Exception as e:
                        print(f"⚠️ Direct LiteRT conversion notice: {e}")

                    if not litert_success:
                        notice_msg = "Google LiteRT runtime currently does not support Vision Transformer deformable attention ops (RT-DETRv2). For LiteRT (.tflite) on mobile/embedded, please select YOLO11 or YOLOv8!"
                        print(f"ℹ️ {notice_msg}")
                        export_results["warnings"].append(notice_msg)

            elif "DEIMv2" in family:
                # 1. Setup DEIMv2 repository
                deim_repo_dir = "/content/DEIMv2"
                if not os.path.exists(deim_repo_dir):
                    print("📥 Cloning Intellindust DEIMv2 repository...")
                    subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/Intellindust-AI-Lab/DEIMv2.git", deim_repo_dir])
                if deim_repo_dir not in sys.path:
                    sys.path.insert(0, deim_repo_dir)

                import math
                from huggingface_hub import hf_hub_download
                from safetensors.torch import load_file as load_safetensors

                # 2. Download pre-trained weights from Hugging Face
                weights_cache_dir = "/content/weights"
                os.makedirs(weights_cache_dir, exist_ok=True)
                ckpt_clean_name = checkpoint.split("/")[-1].lower()
                st_path = os.path.join(weights_cache_dir, f"{ckpt_clean_name}.safetensors")
                pth_path = os.path.join(weights_cache_dir, f"{ckpt_clean_name}.pth")

                if not os.path.exists(pth_path):
                    print(f"📥 Downloading pre-trained DEIMv2 weights from {checkpoint}...")
                    downloaded_st = hf_hub_download(repo_id=checkpoint, filename="model.safetensors")
                    st_data = load_safetensors(downloaded_st)
                    torch.save({"model": st_data}, pth_path)
                    print(f"✅ Successfully downloaded and cached DEIMv2 checkpoint ({os.path.getsize(pth_path)/(1024*1024):.2f} MB)!")

                # 3. Prepare COCO Dataset for DEIMv2
                deim_dir = "/content/deimv2_dataset"
                prepare_coco_dataset(deim_dir, train_samples, val_samples, meta, label2id)

                # 4. Map variant to DEIMv2 config file
                cfg_mapping = {
                    "atto": "deimv2_hgnetv2_atto_coco.yml",
                    "femto": "deimv2_hgnetv2_femto_coco.yml",
                    "pico": "deimv2_hgnetv2_pico_coco.yml",
                    "n_coco": "deimv2_hgnetv2_n_coco.yml",
                    "s_coco": "deimv2_dinov3_s_coco.yml",
                    "m_coco": "deimv2_dinov3_m_coco.yml",
                    "l_coco": "deimv2_dinov3_l_coco.yml",
                    "x_coco": "deimv2_dinov3_x_coco.yml",
                }
                matched_cfg = "deimv2_hgnetv2_n_coco.yml"
                for k, v in cfg_mapping.items():
                    if k in ckpt_clean_name:
                        matched_cfg = v
                        break
                config_path = os.path.join(deim_repo_dir, "configs", "deimv2", matched_cfg)

                weights_dir = "/content/runs/deimv2_model/weights"
                os.makedirs(weights_dir, exist_ok=True)
                last_saved_model_path["path"] = weights_dir
                best_pt = os.path.join(weights_dir, "best.pth")
                last_pt = os.path.join(weights_dir, "last.pth")
                export_results["success"].append("PyTorch (.pth)")

                # 5. Launch DEIMv2 training process
                cmd = [
                    sys.executable, f"{deim_repo_dir}/train.py",
                    "-c", config_path,
                    "-t", pth_path,
                    "--output-dir", "/content/runs/deimv2_model",
                    "-u",
                    f"num_classes={len(id2label)}",
                    "remap_mscoco_category=False",
                    f"epoches={epochs}",
                    f"train_dataloader.dataset.img_folder={deim_dir}/train2017/",
                    f"train_dataloader.dataset.ann_file={deim_dir}/annotations/instances_train2017.json",
                    f"train_dataloader.total_batch_size={batch_size}",
                    f"val_dataloader.dataset.img_folder={deim_dir}/val2017/",
                    f"val_dataloader.dataset.ann_file={deim_dir}/annotations/instances_val2017.json",
                    f"val_dataloader.total_batch_size={batch_size}",
                ]
                if torch.cuda.is_available():
                    cmd.append("--use-amp")

                print(f"🚀 Launching DEIMv2 training ({epochs} epochs, {len(id2label)} classes)...")
                proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, cwd=deim_repo_dir)
                try:
                    current_epoch = 1
                    for line in proc.stdout:
                        print(line, end="")
                        if stop_state.get("stop"):
                            proc.terminate()
                            print("\n🛑 DEIMv2 training stopped cleanly by user.")
                            break

                        ep_match = re.search(r'Epoch:\s*\[(\d+)\]', line)
                        if ep_match:
                            current_epoch = int(ep_match.group(1)) + 1

                        loss_match = re.search(r'loss:\s*([0-9.]+)', line)
                        cur_loss = float(loss_match.group(1)) if loss_match else None

                        val_match = re.search(r'Average Precision\s*\(AP\)\s*@\[\s*IoU=0.50\s*\|[^\]]*\]\s*=\s*([0-9.]+)', line)
                        val_ap = float(val_match.group(1)) if val_match else None

                        if ep_match or loss_match:
                            update_ui_metric(
                                phase=f"Training DEIMv2 (Epoch {current_epoch}/{epochs})",
                                epoch=current_epoch,
                                max_epochs=epochs,
                                train_loss=cur_loss,
                                val_loss=val_ap,
                                append_history=(ep_match is not None)
                            )
                            progress_bar.value = int(45 + 40 * (current_epoch / epochs))
                finally:
                    if proc.stdout:
                        proc.stdout.close()
                    if proc.poll() is None:
                        proc.terminate()
                    proc.wait()

                # Pick up best and last weights (DEIMv2 DetSolver writes best_stg2.pth, best_stg1.pth, best.pth, last.pth)
                deim_out_root = "/content/runs/deimv2_model"
                best_candidates = [
                    os.path.join(deim_out_root, name)
                    for name in ("best_stg2.pth", "best_stg1.pth", "best.pth", "last.pth")
                    if os.path.exists(os.path.join(deim_out_root, name))
                ]
                if best_candidates:
                    shutil.copy2(best_candidates[0], best_pt)
                elif os.path.exists(pth_path):
                    shutil.copy2(pth_path, best_pt)
                deim_last_out = os.path.join(deim_out_root, "last.pth")
                if os.path.exists(deim_last_out):
                    shutil.copy2(deim_last_out, last_pt)

                # 6. Automated Multi-Format Export for DEIMv2
                progress_bar.value = 85
                if os.path.exists(best_pt) and not stop_state["stop"]:
                    if export_onnx:
                        status_text.value = "<b>⏳ Exporting DEIMv2 to ONNX...</b>"
                        update_ui_metric(phase="Exporting ONNX")
                        print("🌐 Exporting DEIMv2 to ONNX format...")
                        try:
                            ensure_packages(["onnx", "onnxslim"])
                            onnx_path = os.path.join(weights_dir, "best.onnx")
                            export_cmd = [
                                sys.executable, f"{deim_repo_dir}/tools/deployment/export_onnx.py",
                                "-c", config_path,
                                "-r", best_pt,
                                "--output-name", onnx_path
                            ]
                            subprocess.check_call(export_cmd, cwd=deim_repo_dir)
                            print("✅ Successfully exported DEIMv2 ONNX model!")
                            export_results["success"].append("ONNX (.onnx)")
                        except Exception as e:
                            print(f"⚠️ ONNX export notice: {e}")
                            export_results["warnings"].append(f"ONNX: {e}")
                    if export_coreml:
                        export_results["warnings"].append("CoreML: DEIMv2 uses PyTorch (.pth) and ONNX (.onnx) deployment pipelines.")
                    if export_litert:
                        export_results["warnings"].append("LiteRT: DEIMv2 deformable attention uses ONNX/TensorRT deployment rather than TFLite.")

            elif "MMDetection" in family:
                # 1. Prepare COCO Dataset structure
                mmdet_dir = "/content/mmdet_dataset"
                prepare_coco_dataset(mmdet_dir, train_samples, val_samples, meta, label2id)

                # 2. Download MMDetection base model & weights via MIM
                zoo_dir = "/content/mmdet_zoo"
                os.makedirs(zoo_dir, exist_ok=True)
                print(f"📥 Fetching MMDetection config and pre-trained weights for '{checkpoint}' via MIM...")
                subprocess.check_call(["mim", "download", "mmdet", "--config", checkpoint, "--dest", zoo_dir])

                # Locate downloaded config and checkpoint
                downloaded_cfgs = [os.path.join(zoo_dir, f) for f in os.listdir(zoo_dir) if f.endswith(".py") and checkpoint in f]
                downloaded_pths = [os.path.join(zoo_dir, f) for f in os.listdir(zoo_dir) if f.endswith(".pth") and checkpoint in f]
                base_cfg_path = downloaded_cfgs[0] if downloaded_cfgs else os.path.join(zoo_dir, f"{checkpoint}.py")
                base_pth_path = downloaded_pths[0] if downloaded_pths else None

                # 3. Create custom MMDetection training configuration
                custom_cfg_path = "/content/mmdet_custom_cfg.py"
                classes_tuple_str = repr(tuple(class_names))
                cfg_script = f"""
_base_ = '{base_cfg_path}'
data_root = '{mmdet_dir}/'
metainfo = dict(classes={classes_tuple_str})

train_dataloader = dict(
    batch_size={batch_size},
    dataset=dict(
        data_root=data_root,
        metainfo=metainfo,
        ann_file='annotations/instances_train2017.json',
        data_prefix=dict(img='train2017/')
    )
)
val_dataloader = dict(
    batch_size={batch_size},
    dataset=dict(
        data_root=data_root,
        metainfo=metainfo,
        ann_file='annotations/instances_val2017.json',
        data_prefix=dict(img='val2017/')
    )
)
val_evaluator = dict(ann_file=data_root + 'annotations/instances_val2017.json')
test_evaluator = val_evaluator
train_cfg = dict(max_epochs={epochs}, val_interval=1)
"""
                if base_pth_path:
                    cfg_script += f"\nload_from = '{base_pth_path}'\n"
                cfg_script += "\nwork_dir = '/content/runs/mmdet_model'\n"

                with open(custom_cfg_path, "w", encoding="utf-8") as f:
                    f.write(cfg_script)

                weights_dir = "/content/runs/mmdet_model/weights"
                os.makedirs(weights_dir, exist_ok=True)
                last_saved_model_path["path"] = weights_dir
                best_pt = os.path.join(weights_dir, "best.pth")
                last_pt = os.path.join(weights_dir, "last.pth")
                export_results["success"].append("PyTorch (.pth)")

                # 4. Train MMDetection using MMEngine Runner
                from mmengine.config import Config
                from mmengine.runner import Runner
                from mmengine.hooks import Hook

                def _override_mmdet_num_classes(node, n_cls):
                    if isinstance(node, dict):
                        if "num_classes" in node:
                            node["num_classes"] = n_cls
                        for v in node.values():
                            _override_mmdet_num_classes(v, n_cls)
                    elif isinstance(node, (list, tuple)):
                        for item in node:
                            _override_mmdet_num_classes(item, n_cls)

                class MMDetectionDashboardHook(Hook):
                    def after_train_iter(self, runner, batch_idx, data_batch=None, outputs=None):
                        if stop_state.get("stop"):
                            runner.should_stop = True
                    def after_train_epoch(self, runner):
                        ep = runner.epoch + 1
                        t_loss = runner.message_hub.get_scalar('train/loss').current() if runner.message_hub.has_scalar('train/loss') else None
                        v_map = runner.message_hub.get_scalar('coco/bbox_mAP_50').current() if runner.message_hub.has_scalar('coco/bbox_mAP_50') else None
                        update_ui_metric(
                            phase=f"Training MMDetection (Epoch {ep}/{epochs})",
                            epoch=ep,
                            max_epochs=epochs,
                            train_loss=float(t_loss) if t_loss is not None else None,
                            val_loss=float(v_map) if v_map is not None else None,
                            append_history=True
                        )
                        progress_bar.value = int(45 + 40 * (ep / epochs))

                print(f"🚀 Starting MMDetection training ({epochs} epochs, {len(class_names)} classes)...")
                mm_cfg = Config.fromfile(custom_cfg_path)
                _override_mmdet_num_classes(mm_cfg.model, len(class_names))
                mm_cfg.dump(custom_cfg_path)
                mm_cfg.custom_hooks = [dict(type=MMDetectionDashboardHook)]
                runner = Runner.from_cfg(mm_cfg)
                runner.train()

                # Find saved checkpoint
                work_dir = "/content/runs/mmdet_model"
                pth_files = sorted([os.path.join(work_dir, f) for f in os.listdir(work_dir) if f.endswith(".pth")], key=os.path.getmtime)
                best_pth_files = [f for f in pth_files if "best" in os.path.basename(f)]
                if best_pth_files:
                    shutil.copy2(best_pth_files[-1], best_pt)
                elif pth_files:
                    shutil.copy2(pth_files[-1], best_pt)
                if pth_files:
                    shutil.copy2(pth_files[-1], last_pt)

                # 5. Automated Multi-Format Export for MMDetection
                progress_bar.value = 85
                if os.path.exists(best_pt) and not stop_state["stop"]:
                    if export_onnx:
                        status_text.value = "<b>⏳ Exporting MMDetection to ONNX...</b>"
                        update_ui_metric(phase="Exporting ONNX")
                        print("🌐 Exporting MMDetection to ONNX format...")
                        try:
                            ensure_packages(["onnx", "onnxslim"])
                            onnx_path = os.path.join(weights_dir, "best.onnx")
                            val_imgs_list = os.listdir(f"{mmdet_dir}/val2017")
                            sample_img = f"{mmdet_dir}/val2017/{val_imgs_list[0]}"
                            export_cmd = [
                                "mim", "run", "mmdet", "pytorch2onnx",
                                custom_cfg_path,
                                best_pt,
                                "--output-file", onnx_path,
                                "--input-img", sample_img
                            ]
                            subprocess.check_call(export_cmd)
                            print("✅ Successfully exported MMDetection ONNX model!")
                            export_results["success"].append("ONNX (.onnx)")
                        except Exception as e:
                            print(f"⚠️ ONNX export notice: {e}")
                            export_results["warnings"].append(f"ONNX: {e}")
                    if export_coreml:
                        export_results["warnings"].append("CoreML: MMDetection models export via PyTorch (.pth) and ONNX (.onnx) / MMDeploy.")
                    if export_litert:
                        export_results["warnings"].append("LiteRT: MMDetection models export via PyTorch (.pth) and ONNX (.onnx) / MMDeploy.")

            elif "YOLOX" in family:
                # 1. Setup Megvii YOLOX environment
                yolox_repo_dir = "/content/YOLOX"
                if not os.path.exists(yolox_repo_dir):
                    print("📥 Cloning Megvii YOLOX repository...")
                    subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/Megvii-BaseDetection/YOLOX.git", yolox_repo_dir])
                if yolox_repo_dir not in sys.path:
                    sys.path.insert(0, yolox_repo_dir)

                import math
                from yolox.exp import get_exp
                from yolox.models.network_blocks import SiLU
                from yolox.utils import replace_module

                # 2. Download pre-trained checkpoint weights if needed
                weights_cache_dir = "/content/weights"
                os.makedirs(weights_cache_dir, exist_ok=True)
                ckpt_path = os.path.join(weights_cache_dir, checkpoint)
                if not os.path.exists(ckpt_path):
                    print(f"📥 Downloading pre-trained {checkpoint} weights from Megvii releases...")
                    ckpt_url = f"https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/{checkpoint}"
                    import urllib.request
                    urllib.request.urlretrieve(ckpt_url, ckpt_path)
                    print(f"✅ Successfully downloaded {checkpoint} ({os.path.getsize(ckpt_path)/(1024*1024):.2f} MB)!")

                # 3. Prepare COCO Dataset structure required by YOLOX
                yolox_dir = "/content/yolox_dataset"
                prepare_coco_dataset(yolox_dir, train_samples, val_samples, meta, label2id)

                # 4. Instantiate and configure YOLOX experiment
                exp_id = checkpoint.replace(".pth", "").replace("_", "-")
                exp = get_exp(None, exp_id)
                exp.data_dir = yolox_dir
                exp.train_ann = "instances_train2017.json"
                exp.val_ann = "instances_val2017.json"
                exp.num_classes = len(id2label)
                exp.max_epoch = epochs
                exp.input_size = (imgsz, imgsz)
                exp.test_size = (imgsz, imgsz)
                exp.basic_lr_per_img = lr / max(1, batch_size)
                exp.output_dir = "/content/runs"
                exp.exp_name = "yolox_model"
                exp.data_num_workers = 2
                exp.eval_interval = 1

                # 5. Build model and load pretrained weights (with automatic custom class head adaptation)
                model = exp.get_model()
                if os.path.exists(ckpt_path):
                    print(f"📦 Loading pre-trained weights from {ckpt_path}...")
                    ckpt_data = torch.load(ckpt_path, map_location="cpu")
                    state_dict = ckpt_data["model"] if isinstance(ckpt_data, dict) and "model" in ckpt_data else ckpt_data
                    model_dict = model.state_dict()
                    filtered_dict = {
                        k: v for k, v in state_dict.items()
                        if k in model_dict and model_dict[k].shape == v.shape
                    }
                    model.load_state_dict(filtered_dict, strict=False)
                    skipped = len(state_dict) - len(filtered_dict)
                    print(f"✅ Loaded {len(filtered_dict)} layers ({skipped} head layers initialized for {len(id2label)} custom classes).")

                device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
                model.to(device)

                # 6. Build data loaders & optimizer
                train_loader = exp.get_data_loader(batch_size=batch_size, is_distributed=False)
                evaluator = exp.get_evaluator(batch_size=batch_size, is_distributed=False)

                exp.model = model
                optimizer = exp.get_optimizer(batch_size)
                scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=lr * 0.05)
                scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

                weights_dir = "/content/runs/yolox_model/weights"
                os.makedirs(weights_dir, exist_ok=True)
                last_saved_model_path["path"] = weights_dir
                best_pt = os.path.join(weights_dir, "best.pth")
                last_pt = os.path.join(weights_dir, "last.pth")
                export_results["success"].append("PyTorch (.pth)")

                iter_loader = iter(train_loader)
                iters_per_epoch = max(1, math.ceil(len(train_samples) / batch_size))
                best_ap50 = 0.0

                print(f"🚀 Starting YOLOX training: {epochs} epochs, batch size {batch_size}, {iters_per_epoch} iterations/epoch...")

                for epoch in range(epochs):
                    if stop_state.get("stop"):
                        print("\n🛑 Training stop requested. Exiting loop cleanly...")
                        break

                    model.train()
                    epoch_losses = []

                    for it in range(iters_per_epoch):
                        if stop_state.get("stop"):
                            break

                        try:
                            inps, targets = next(iter_loader)
                        except StopIteration:
                            iter_loader = iter(train_loader)
                            inps, targets = next(iter_loader)
                        inps = inps.to(device)
                        targets = targets.to(device)

                        optimizer.zero_grad()
                        if torch.cuda.is_available():
                            with torch.cuda.amp.autocast():
                                outputs = model(inps, targets)
                                loss = outputs["total_loss"]
                            scaler.scale(loss).backward()
                            scaler.step(optimizer)
                            scaler.update()
                        else:
                            outputs = model(inps, targets)
                            loss = outputs["total_loss"]
                            loss.backward()
                            optimizer.step()

                        epoch_losses.append(loss.item())

                    if stop_state.get("stop"):
                        break

                    scheduler.step()
                    avg_loss = sum(epoch_losses) / len(epoch_losses) if epoch_losses else 0.0

                    # Validation evaluation
                    val_metric = None
                    try:
                        model.eval()
                        with torch.no_grad():
                            eval_res = evaluator.evaluate(model, distributed=False, half=False)
                            if isinstance(eval_res, tuple) and len(eval_res) >= 2:
                                ap50_95, ap50 = eval_res[0], eval_res[1]
                                val_metric = float(ap50)
                                if val_metric >= best_ap50:
                                    best_ap50 = val_metric
                                    torch.save({"model": model.state_dict(), "num_classes": len(id2label), "names": id2label}, best_pt)
                    except Exception as eval_err:
                        print(f"⚠️ Validation note (Epoch {epoch+1}): {eval_err}")

                    # Always save latest
                    torch.save({"model": model.state_dict(), "num_classes": len(id2label), "names": id2label}, last_pt)
                    if not os.path.exists(best_pt):
                        torch.save({"model": model.state_dict(), "num_classes": len(id2label), "names": id2label}, best_pt)

                    ep = epoch + 1
                    update_ui_metric(
                        phase=f"Training (Epoch {ep}/{epochs})",
                        epoch=ep,
                        max_epochs=epochs,
                        train_loss=float(avg_loss),
                        val_loss=val_metric,
                        append_history=True
                    )
                    progress_bar.value = int(45 + 40 * (ep / epochs))

                # 7. Automated Multi-Format Export for YOLOX
                progress_bar.value = 85
                if os.path.exists(best_pt) and not stop_state["stop"]:
                    export_model = exp.get_model()
                    ckpt_save = torch.load(best_pt, map_location="cpu")
                    export_model.load_state_dict(ckpt_save["model"] if "model" in ckpt_save else ckpt_save)
                    export_model.eval()
                    export_model = replace_module(export_model, torch.nn.SiLU, SiLU)
                    export_model.head.decode_in_inference = True
                    dummy_input = torch.randn(1, 3, imgsz, imgsz)

                    if export_onnx:
                        status_text.value = "<b>⏳ Exporting YOLOX to ONNX...</b>"
                        update_ui_metric(phase="Exporting ONNX")
                        print("🌐 Exporting to ONNX format...")
                        try:
                            ensure_packages(["onnx", "onnxslim"])
                            onnx_path = os.path.join(weights_dir, "best.onnx")
                            torch.onnx.export(
                                export_model,
                                dummy_input,
                                onnx_path,
                                input_names=["images"],
                                output_names=["output"],
                                dynamic_axes={"images": {0: "batch"}, "output": {0: "batch"}},
                                opset_version=12,
                            )
                            try:
                                import onnxslim
                                onnxslim.slim(onnx_path, onnx_path)
                            except Exception:
                                pass
                            print("✅ Successfully exported ONNX model!")
                            export_results["success"].append("ONNX (.onnx)")
                        except Exception as e:
                            print(f"⚠️ ONNX export notice: {e}")
                            export_results["warnings"].append(f"ONNX: {e}")

                    if export_coreml:
                        status_text.value = "<b>⏳ Exporting YOLOX to Apple CoreML...</b>"
                        update_ui_metric(phase="Exporting CoreML")
                        print("📱 Exporting to Apple CoreML (.mlpackage)...")
                        try:
                            ensure_packages(["coremltools"])
                            import coremltools as ct
                            traced = torch.jit.trace(export_model, dummy_input)
                            mlmodel = ct.convert(
                                traced,
                                inputs=[ct.TensorType(name="images", shape=(1, 3, imgsz, imgsz), scale=1/255.0)]
                            )
                            coreml_path = os.path.join(weights_dir, "best.mlpackage")
                            mlmodel.save(coreml_path)
                            print("✅ Successfully exported Apple CoreML model!")
                            export_results["success"].append("Apple CoreML (.mlpackage)")
                        except Exception as e:
                            print(f"⚠️ CoreML export notice: {e}")
                            export_results["warnings"].append(f"CoreML: {e}")

                    if export_litert:
                        status_text.value = "<b>⏳ Exporting YOLOX to Google LiteRT (TFLite)...</b>"
                        update_ui_metric(phase="Exporting LiteRT")
                        print("🤖 Exporting to Google LiteRT (.tflite)...")
                        try:
                            ensure_packages(["ai-edge-litert>=2.1.4", "litert-torch>=0.9.0"])
                            import litert_torch
                            edge_model = litert_torch.convert(export_model, (dummy_input,))
                            tflite_path = os.path.join(weights_dir, "best.tflite")
                            edge_model.export(tflite_path)
                            print("✅ Successfully exported Google LiteRT model!")
                            export_results["success"].append("Google LiteRT (.tflite)")
                        except Exception as e:
                            print(f"⚠️ LiteRT export notice: {e}")
                            export_results["warnings"].append(f"LiteRT: {e}")

            else:
                import yaml
                from ultralytics import YOLO

                yolo_dir = "/content/yolo_dataset"
                if os.path.exists(yolo_dir):
                    shutil.rmtree(yolo_dir)

                for split, split_samples in [("train", train_samples), ("val", val_samples)]:
                    os.makedirs(f"{yolo_dir}/images/{split}", exist_ok=True)
                    os.makedirs(f"{yolo_dir}/labels/{split}", exist_ok=True)
                    for sample in split_samples:
                        ann = sly.Annotation.load_json_file(sample["ann_path"], meta)
                        ih, iw = ann.img_size
                        stem = f"{sample['dataset_name']}_{Path(sample['item_name']).stem}"
                        ext = Path(sample["item_name"]).suffix
                        shutil.copy2(sample["img_path"], f"{yolo_dir}/images/{split}/{stem}{ext}")

                        lines = []
                        for l in ann.labels:
                            if l.obj_class.name not in label2id:
                                continue
                            cid = label2id[l.obj_class.name]
                            bb = l.geometry.to_bbox()
                            if bb.width <= 0 or bb.height <= 0 or iw <= 0 or ih <= 0:
                                continue
                            xc = min(max(((bb.left + bb.right) / 2.0) / iw, 0.0), 1.0)
                            yc = min(max(((bb.top + bb.bottom) / 2.0) / ih, 0.0), 1.0)
                            w = min(max(bb.width / iw, 0.0), 1.0)
                            h = min(max(bb.height / ih, 0.0), 1.0)
                            lines.append(f"{cid} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")

                        with open(f"{yolo_dir}/labels/{split}/{stem}.txt", "w", encoding="utf-8") as f:
                            f.write("\n".join(lines))

                yaml_path = f"{yolo_dir}/dataset.yaml"
                with open(yaml_path, "w", encoding="utf-8") as f:
                    yaml.dump({"path": yolo_dir, "train": "images/train", "val": "images/val", "names": id2label}, f)

                model = YOLO(checkpoint)
                if batch_dropdown.value == -1 and torch.cuda.is_available():
                    try:
                        from ultralytics.utils.autobatch import check_train_batch_size
                        probed_bs = int(check_train_batch_size(model.model, imgsz=imgsz, amp=True))
                        if probed_bs >= 2:
                            max_sensible = max(2, int(2 ** math.floor(math.log2(max(2, len(train_samples))))))
                            batch_size = max(2, min(64, probed_bs, max_sensible))
                            print(f"🔬 Ultralytics GPU Memory Profiler refined Auto-Batch to batch_size={batch_size}")
                    except Exception:
                        pass

                # Ultralytics Callbacks for Stop & Live Dashboard
                def on_train_batch_start(trainer):
                    if stop_state.get("stop"):
                        trainer.stop = True

                def on_fit_epoch_end(trainer):
                    ep = trainer.epoch + 1
                    t_loss = getattr(trainer, "loss", None)
                    v_loss = trainer.metrics.get("metrics/precision(B)", None) if hasattr(trainer, 'metrics') else None
                    update_ui_metric(
                        phase=f"Training (Epoch {ep}/{trainer.epochs})",
                        epoch=ep,
                        max_epochs=trainer.epochs,
                        train_loss=float(t_loss) if t_loss is not None else None,
                        val_loss=float(v_loss) if v_loss is not None else None,
                        append_history=True
                    )
                    progress_bar.value = int(45 + 40 * (ep / trainer.epochs))

                model.add_callback("on_train_batch_start", on_train_batch_start)
                model.add_callback("on_fit_epoch_end", on_fit_epoch_end)

                model.train(data=yaml_path, epochs=epochs, imgsz=imgsz, batch=batch_size, lr0=lr, project="/content/runs", name="ultralytics_model", exist_ok=True)
                
                weights_dir = "/content/runs/ultralytics_model/weights"
                last_saved_model_path["path"] = weights_dir
                best_pt = os.path.join(weights_dir, "best.pt")
                export_results["success"].append("PyTorch (.pt)")

                # 5. Automated Multi-Format Export for Ultralytics
                progress_bar.value = 85
                if os.path.exists(best_pt) and not stop_state["stop"]:
                    exp_model = YOLO(best_pt)

                    if export_onnx:
                        status_text.value = "<b>⏳ Exporting YOLO to ONNX...</b>"
                        update_ui_metric(phase="Exporting ONNX")
                        print("🌐 Exporting to ONNX format...")
                        try:
                            ensure_packages(["onnx", "onnxslim"])
                            exp_model.export(format="onnx", imgsz=imgsz, dynamic=True)
                            print("✅ Successfully exported ONNX model!")
                            export_results["success"].append("ONNX (.onnx)")
                        except Exception as e:
                            print(f"⚠️ ONNX export notice: {e}")
                            export_results["warnings"].append(f"ONNX: {e}")

                    if export_coreml:
                        status_text.value = "<b>⏳ Exporting YOLO to Apple CoreML...</b>"
                        update_ui_metric(phase="Exporting CoreML")
                        print("📱 Exporting to Apple CoreML (.mlpackage)...")
                        try:
                            ensure_packages(["coremltools"])
                            exp_model.export(format="coreml", imgsz=imgsz, nms=True)
                            print("✅ Successfully exported Apple CoreML model!")
                            export_results["success"].append("Apple CoreML (.mlpackage)")
                        except Exception as e:
                            print(f"⚠️ CoreML export notice: {e}")
                            export_results["warnings"].append(f"CoreML: {e}")

                    if export_litert:
                        status_text.value = "<b>⏳ Exporting YOLO to Google LiteRT (TFLite)...</b>"
                        update_ui_metric(phase="Exporting LiteRT")
                        print("🤖 Exporting to Google LiteRT (.tflite)...")
                        try:
                            ensure_packages(["ai-edge-litert>=2.1.4", "litert-torch>=0.9.0"])
                            try:
                                exp_model.export(format="litert", imgsz=imgsz)
                            except Exception:
                                exp_model.export(format="tflite", imgsz=imgsz)
                            print("✅ Successfully exported Google LiteRT model!")
                            export_results["success"].append("Google LiteRT (.tflite)")
                        except Exception as e:
                            print(f"⚠️ LiteRT export notice: {e}")
                            export_results["warnings"].append(f"LiteRT: {e}")

            progress_bar.value = 100
            progress_bar.bar_style = "success"
            final_status = "Training stopped cleanly by user." if stop_state["stop"] else "Training & exports completed!"
            
            succ_pills = "".join([f"<span style='background:#dcfce7; color:#15803d; padding:2px 8px; border-radius:12px; font-size:11px; font-weight:700; margin-right:4px;'>✅ {s}</span>" for s in export_results["success"]])
            warn_pills = ""
            if export_results["warnings"]:
                warn_pills = "<div style='margin-top:8px; background:#fffbeb; border:1px solid #fde68a; border-radius:6px; padding:8px 10px; font-size:12px; color:#b45309;'>" + "".join([f"⚠️ <b>Notice:</b> {w}<br>" for w in export_results["warnings"]]) + "</div>"

            status_text.value = f"""
            <div style='background:#f0fdf4; border:1px solid #86efac; border-radius:8px; padding:12px 14px; margin-top:8px;'>
                <div style='color:#166534; font-weight:700; font-size:13px;'>✅ {final_status}</div>
                <div style='margin-top:6px; font-size:12px; color:#334155;'><b>Exported Formats:</b> {succ_pills}</div>
                {warn_pills}
            </div>
            """
            update_ui_metric(phase="Completed" if not stop_state["stop"] else "Stopped Cleanly")
            btn_download_zip.disabled = False

        except Exception as e:
            progress_bar.bar_style = "danger"
            status_text.value = f"<b style='color:#dc2626;'>❌ Error: {e}</b>"
            update_ui_metric(phase="Error Occurred")
            raise
        finally:
            pipeline_busy["busy"] = False
            btn_train.disabled = False
            btn_stop.disabled = True

def start_training_thread(_=None):
    if pipeline_busy["busy"]:
        return
    # Launch training in a dedicated daemon thread.
    # This keeps the main IPython event loop completely free and responsive,
    # allowing instant switching between Visual Dashboard and Raw Console,
    # as well as immediate response to Stop Training clicks!
    threading.Thread(target=run_training_pipeline, daemon=True).start()

btn_train.on_click(start_training_thread)

zip_download_state = {
    "busy": False,
    "last_download_time": 0.0
}

def download_zip(_):
    now = time.time()
    # Enforce strict 12-second debounce and busy lock to completely eliminate duplicate downloads
    if zip_download_state["busy"] or (now - zip_download_state["last_download_time"] < 12.0):
        return

    p = last_saved_model_path["path"]
    if not p or not os.path.exists(p):
        return

    zip_download_state["busy"] = True
    zip_download_state["last_download_time"] = now

    btn_download_zip.disabled = True
    btn_download_zip.button_style = "warning"
    btn_download_zip.description = "⏳ Packaging & Downloading..."

    status_text.value = """
    <div style='background:#f0fdf4; border:1px solid #86efac; border-radius:8px; padding:10px 14px; margin-top:8px;'>
        <div style='color:#166534; font-weight:700; font-size:13px;'>💾 Download Started!</div>
        <div style='font-size:12px; color:#334155; margin-top:4px;'>Packaging complete. Google Colab is sending <code>trained_model_bundle.zip</code> to your browser. Please check your browser's download prompt.</div>
    </div>
    """

    try:
        zip_path = shutil.make_archive("/content/trained_model_bundle", "zip", p)
        with log_output:
            print(f"📦 Archive generated: {zip_path}")
            print("📋 Bundle contents:")
            for root, dirs, files in os.walk(p):
                for file in files:
                    rel = os.path.relpath(os.path.join(root, file), p)
                    sz_mb = os.path.getsize(os.path.join(root, file)) / (1024 * 1024)
                    print(f"  • {rel} ({sz_mb:.2f} MB)")

        btn_download_zip.description = "✅ Download Triggered!"
        btn_download_zip.button_style = "success"

        try:
            from google.colab import files
            files.download(zip_path)
        except Exception:
            with log_output:
                print(f"💾 File ready on disk: {zip_path}")

    finally:
        # Keep button locked for a 12-second cooldown to completely prevent duplicate downloads
        def reset_download_cooldown():
            time.sleep(12)
            btn_download_zip.disabled = False
            btn_download_zip.button_style = "primary"
            btn_download_zip.description = "💾 Re-download Model Bundle (.zip)"
            zip_download_state["busy"] = False

        threading.Thread(target=reset_download_cooldown, daemon=True).start()

btn_download_zip.on_click(download_zip)

# --- 5. RENDER DASHBOARD ---
card_layout = widgets.Layout(
    border="1px solid #e2e8f0",
    padding="16px",
    margin="0 0 12px 0",
    border_radius="10px",
    box_shadow="0 1px 3px 0 rgba(0, 0, 0, 0.05)"
)

def _section_header_html(icon_str, title_str, subtitle_str=""):
    sub_html = f"<span style='font-size:11px; color:#94a3b8; font-weight:500; margin-left:auto;'>{subtitle_str}</span>" if subtitle_str else ""
    return widgets.HTML(
        f"<div style='background:linear-gradient(90deg, #1e293b, #334155); color:#f8fafc; padding:8px 14px; border-radius:8px; display:flex; align-items:center; gap:8px; margin-bottom:10px;'>"
        f"<span style='font-size:15px;'>{icon_str}</span>"
        f"<span style='margin:0; font-size:13px; font-weight:700; letter-spacing:0.01em; color:#f8fafc;'>{title_str}</span>"
        f"{sub_html}"
        f"</div>"
    )

sec1 = widgets.VBox([
    _section_header_html("1️⃣", "Supervisely Connection & Project Discovery", "Auto-detects validation datasets"),
    server_input,
    token_row,
    project_dropdown,
    project_status_html
], layout=card_layout)

sec2 = widgets.VBox([
    _section_header_html("2️⃣", "Model Architecture & License Selection", "10 Families • 54 Pretrained Checkpoints"),
    family_dropdown,
    checkpoint_dropdown,
    model_info_html
], layout=card_layout)

sec3 = widgets.VBox([
    _section_header_html("3️⃣", "Training Hyperparameters & Export Targets", "PyTorch • ONNX • CoreML • LiteRT"),
    epochs_slider,
    widgets.HBox([batch_dropdown, imgsz_dropdown]),
    lr_dropdown,
    val_split_slider,
    val_dataset_notice_html,
    training_calc_html,
    widgets.HTML("<div style='margin-top:12px; margin-bottom:4px; background:#f1f5f9; border:1px solid #cbd5e1; padding:6px 12px; border-radius:6px; font-weight:700; font-size:12px; color:#1e293b;'>📦 Target Export Formats (Automatically converted & packaged into .zip):</div>"),
    export_formats_row
], layout=card_layout)

sec4 = widgets.VBox([
    _section_header_html("4️⃣", "Training Execution & Live Visual Monitor", "Real-time KPIs, ETA & Convergence Drill"),
    widgets.HBox([btn_train, btn_stop, btn_download_zip], layout=widgets.Layout(gap="8px", margin="0 0 8px 0")),
    progress_bar,
    status_text,
    view_toggle,
    view_container
], layout=card_layout)

display(widgets.VBox([header_html, sec1, sec2, sec3, sec4]))

if default_token:
    fetch_projects()